# 04c · 고정 test 추론과 최종 내부 평가

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** GPU의 익명 추론과 CPU의 private 평가를 분리하고 예측 hash 검증 후 전체·유형·장애물 RMSE를 계산합니다.
- **입력:** 04B selection과 가중치, 익명 test shard, 봉인된 예측, private truth·장애물 metadata
- **출력:** prediction seal, RMSE/구간별 평가와 evaluation seal
- **실제 저장된 실행 상태:** 저장 출력은 PRIVATE EVALUATION COMPLETE, winner=v4-fixed-router-multitask, overall RMSE=0.14233391569554288 m입니다. 원 코드의 claim_scope는 internal_fixed_holdout_after_prior_truth_exposure입니다. 이전 실험에서 정답이 노출된 내부 고정 holdout의 후속 비교이며 외부 공식 평가나 새 미관측 test 성능으로 주장하지 않습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/04c_final_evaluation.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/04c_final_evaluation.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 04C. 고정 type router 익명 test 추론과 최종 평가

같은 노트북을 두 번 분리 실행합니다. 먼저 GPU 런타임에서 RUN_PHASE = "INFERENCE"로
selection seal과 익명 test image seal만 읽어 예측을 봉인합니다. 라우터는 sample_id 첫
문자로 B/G/S를 224px full-frame 경로에, A/C/E/F/H를 공통 YOLO26s와 384px ROI 경로에
고정 배정하고, detector 미검출에는 full-frame 예측을 사용합니다. 이어서 런타임을 초기화한
CPU 세션에서 RUN_PHASE = "EVALUATION"으로 바꾸고, 사전에 고정된 평가 계획과 private
metadata를 이용해 전체·유형·장애물·유형×장애물 RMSE 및 bootstrap 신뢰구간을 계산합니다.

현재 test 정답은 이전 분석에서 이미 접근한 고정 holdout이므로 결과의 주장 범위는
새로운 비공개 리더보드 추정치가 아니라 내부 재현 평가입니다. 추론 단계에는 장애물 정답과
수위 정답이 입력되지 않으며, 평가 단계는 checkpoint를 다시 읽거나 재추론하지 않습니다.


## Configuration: edit only this cell before each clean Colab run.


In [ ]:
from __future__ import annotations

import importlib.metadata
import importlib.util
import os
import sys
import subprocess

DEPENDENCY_MODE = os.environ.get(
    "DCC_V4_INSTALL_DEPENDENCIES", "auto"
).strip().lower()
if DEPENDENCY_MODE not in {"auto", "true", "false"}:
    raise ValueError("DCC_V4_INSTALL_DEPENDENCIES must be auto, true, or false")
REQUIRED_MODULES = [
    "google.cloud.storage",
    "pandas",
    "pyarrow",
    "sklearn",
    "matplotlib",
    "ultralytics",
]
missing_modules = []
for module_name in REQUIRED_MODULES:
    try:
        available = importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        available = False
    if available and module_name == "ultralytics":
        try:
            available = importlib.metadata.version("ultralytics") == "8.4.121"
        except importlib.metadata.PackageNotFoundError:
            available = False
    if not available:
        missing_modules.append(module_name)
INSTALL_DEPENDENCIES = DEPENDENCY_MODE == "true" or (
    DEPENDENCY_MODE == "auto" and bool(missing_modules)
)
if INSTALL_DEPENDENCIES:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "google-cloud-storage>=2.19,<4",
            "pandas>=2.2,<3",
            "pyarrow>=18,<24",
            "scikit-learn>=1.4,<2",
            "matplotlib>=3.8,<4",
            "ultralytics==8.4.121",
        ]
    )

RUN_PHASE = os.environ.get("DCC_V4_RUN_PHASE", "EVALUATION").upper()
if RUN_PHASE not in {"INFERENCE", "EVALUATION"}:
    raise ValueError("DCC_V4_RUN_PHASE must be INFERENCE or EVALUATION")

PROJECT_ID = os.environ.get(
    "DCC_PROJECT_ID", "YOUR_GCP_PROJECT_ID"
)
BUCKET_NAME = os.environ.get(
    "DCC_BUCKET_NAME", "YOUR_GCS_BUCKET"
)

# Frozen 04B selection seal.
SELECTION_SEAL_URI = os.environ.get(
    "DCC_V4_SELECTION_SEAL_URI",
    f"gs://{BUCKET_NAME}/dcc/mission3/v4/selection/"
    "m3-selection-v4-router-fixedtest-v4/model_selection_seal.json",
)

# This reuses the already materialized anonymous sample set.  It does not make
# the resulting score blind again; CLAIM_SCOPE records that limitation.
INFERENCE_IMAGE_SEAL_URI = os.environ.get(
    "DCC_V4_INFERENCE_IMAGE_SEAL_URI",
    f"gs://{BUCKET_NAME}/dcc/inference/mission3/m3ds-v2-pilot/test_image_seal.json",
)
EXPECTED_INFERENCE_DATASET_VERSION = "m3ds-v2-pilot"

PREDICTION_RUN_ID = os.environ.get(
    "DCC_V4_PREDICTION_RUN_ID", "m3-v4-router-obstacle-test-v4"
)
PREDICTION_OUTPUT_PREFIX = (
    f"dcc/inference/mission3/v4/predictions/{PREDICTION_RUN_ID}"
)
PREDICTION_SEAL_URI = os.environ.get(
    "DCC_V4_PREDICTION_SEAL_URI",
    f"gs://{BUCKET_NAME}/{PREDICTION_OUTPUT_PREFIX}/prediction_seal.json",
)

EVALUATION_ID = os.environ.get(
    "DCC_V4_EVALUATION_ID", "m3-v4-router-obstacle-eval-v4"
)
EVALUATION_PREFIX = f"dcc/eval_private/mission3/v4/{EVALUATION_ID}"

# The previous private test was already inspected.  Keep this exact declaration
# unless a genuinely new, untouched external holdout is supplied.
CLAIM_SCOPE = os.environ.get(
    "DCC_V4_CLAIM_SCOPE", "internal_fixed_holdout_after_prior_truth_exposure"
)
TEST_USE_POLICY = os.environ.get("DCC_V4_TEST_USE_POLICY", "research_repeat")
if TEST_USE_POLICY not in {"research_repeat", "strict_single_use"}:
    raise ValueError("unsupported DCC_V4_TEST_USE_POLICY")
if CLAIM_SCOPE.startswith("blind") and TEST_USE_POLICY != "strict_single_use":
    raise AssertionError("blind claim requires strict_single_use")

# The URI is public contract metadata; the referenced Parquet remains private.
# INFERENCE never loads either object.  EVALUATION pins the contract generation
# and SHA before it freezes the evaluation plan.
PRIVATE_SPLIT_PREFIX = "dcc/eval_private/mission3/v4/v4_ml_fixedtest_v2/"
PRIVATE_TEST_EVAL_CONTRACT_URI = os.environ.get(
    "DCC_V4_PRIVATE_TEST_EVAL_CONTRACT_URI",
    f"gs://{BUCKET_NAME}/{PRIVATE_SPLIT_PREFIX}"
    "test_eval_contract_v4_ml_fixedtest_v2.json",
)
if RUN_PHASE == "INFERENCE" and os.environ.get(
    "DCC_V4_PRIVATE_TEST_EVAL_CONTRACT_JSON", ""
):
    raise AssertionError("inference runtime must not receive private contract content")

BATCH_SIZE = int(os.environ.get("DCC_V4_BATCH_SIZE", "64"))
DETECTOR_BATCH_SIZE = int(os.environ.get("DCC_V4_DETECTOR_BATCH_SIZE", "4"))
NUM_WORKERS = int(os.environ.get("DCC_V4_NUM_WORKERS", "2"))
BOOTSTRAP_REPS = int(os.environ.get("DCC_V4_BOOTSTRAP_REPS", "500"))
BOOTSTRAP_SEED = 20260902
MIN_OBSTACLE_SLICE_N = 10

TYPE_ORDER = list("ABCEFGHS")
TYPE_TO_INDEX = {name: index for index, name in enumerate(TYPE_ORDER)}
FIXED_ROUTE_BY_TYPE = {
    "A": "yolo_roi",
    "B": "full_frame",
    "C": "yolo_roi",
    "E": "yolo_roi",
    "F": "yolo_roi",
    "G": "full_frame",
    "H": "yolo_roi",
    "S": "full_frame",
}
EXPECTED_ROUTING_RULE = {
    "routing_key": "type_from_sample_id_first_character",
    "route_by_type": FIXED_ROUTE_BY_TYPE,
    "components": ["full_frame", "yolo_roi"],
}
EXPECTED_ROUTE_COUNTS = {"full_frame": 6833, "yolo_roi": 23976}
EXPECTED_YOLO_INFERENCE_RULE = {
    "detector_confidence": 0.05,
    "max_detections": 10,
    "detector_imgsz": 960,
    "detector_iou": 0.7,
    "crop_padding": 0.15,
    "meter_fusion": "confidence_weighted_median",
    "obstacle_fusion": "confidence_weighted_mean",
    "detector_miss_fallback": "full_frame_regressor",
}
OBSTACLE_ORDER = [
    "거센물살",
    "거울반사",
    "기타",
    "달빛반사",
    "반투명",
    "부유물",
    "부착물",
    "사람",
    "수풀",
    "안개",
    "탁류",
    "표면손상",
    "표면오염",
    "야간",
    "와류",
]
DEFAULT_TYPE_RANGES = {
    "A": (0.0, 15.0),
    "B": (0.0, 9.0),
    "C": (0.0, 15.0),
    "E": (0.0, 10.0),
    "F": (0.0, 4.0),
    "G": (0.0, 4.0),
    "H": (0.0, 20.0),
    "S": (0.0, 20.0),
}

LOCAL_ROOT = os.environ.get(
    "DCC_V4_LOCAL_ROOT",
    "/content/dcc_mission3_v4_inference"
    if RUN_PHASE == "INFERENCE"
    else "/content/dcc_mission3_v4_evaluation",
)


## Imports and deterministic runtime setup.


In [ ]:
import gc
import hashlib
import io
import json
import math
import platform
import re
import shutil
import tarfile
from collections import defaultdict
from collections.abc import Iterable, Mapping, Sequence
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torchvision
from google.api_core.exceptions import PreconditionFailed
from google.cloud import storage
from PIL import Image
from sklearn.metrics import average_precision_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import ultralytics
from ultralytics import YOLO

if ultralytics.__version__ != "8.4.121":
    raise RuntimeError(
        "frozen YOLO26s contract requires ultralytics==8.4.121"
    )

try:
    from google.colab import auth
except ImportError:  # Allows static/local verification outside Colab.
    auth = None

LOCAL_ROOT = Path(LOCAL_ROOT)
LOCAL_INPUT = LOCAL_ROOT / "input"
LOCAL_SHARDS = LOCAL_ROOT / "shards"
LOCAL_IMAGES = LOCAL_ROOT / "anonymous_images"
LOCAL_ROUTED_IMAGES = LOCAL_ROOT / "routed_detector_images"
LOCAL_OUTPUT = LOCAL_ROOT / "output"
LOCAL_META = LOCAL_ROOT / "meta"
LOCAL_FIGURES = LOCAL_ROOT / "figures"
for _path in [
    LOCAL_INPUT,
    LOCAL_SHARDS,
    LOCAL_IMAGES,
    LOCAL_ROUTED_IMAGES,
    LOCAL_OUTPUT,
    LOCAL_META,
    LOCAL_FIGURES,
]:
    _path.mkdir(parents=True, exist_ok=True)

torch.manual_seed(BOOTSTRAP_SEED)
np.random.seed(BOOTSTRAP_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BOOTSTRAP_SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Cryptographic and GCS artifact helpers.


In [ ]:
HEX64 = re.compile(r"^[0-9a-f]{64}$")
DESCRIPTOR_KEYS = {"uri", "generation", "bytes", "sha256", "artifact_type"}


def canonical_sha256(value: Any) -> str:
    raw = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def sha256_file(path: str | Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def parse_gs_uri(uri: str) -> tuple[str, str]:
    if not str(uri).startswith("gs://"):
        raise ValueError(f"not a GCS URI: {uri!r}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    if not bucket_name or not object_name:
        raise ValueError(f"invalid GCS URI: {uri!r}")
    return bucket_name, object_name


def require_all_assertions(payload: Mapping[str, Any], label: str) -> None:
    assertions = payload.get("assertions")
    if not isinstance(assertions, Mapping) or not assertions:
        raise AssertionError(f"{label}: assertions missing")
    failed = [key for key, value in assertions.items() if not bool(value)]
    if failed:
        raise AssertionError(f"{label}: failed assertions: {failed}")


def validate_descriptor_shape(descriptor: Mapping[str, Any], label: str) -> None:
    missing = DESCRIPTOR_KEYS - set(descriptor)
    if missing:
        raise AssertionError(f"{label}: descriptor keys missing: {sorted(missing)}")
    if int(descriptor["generation"]) <= 0:
        raise AssertionError(f"{label}: invalid generation")
    if int(descriptor["bytes"]) < 0:
        raise AssertionError(f"{label}: invalid byte count")
    if not HEX64.fullmatch(str(descriptor["sha256"])):
        raise AssertionError(f"{label}: invalid SHA-256")
    parse_gs_uri(str(descriptor["uri"]))


def authenticate_storage() -> tuple[storage.Client, storage.Bucket]:
    if auth is not None:
        auth.authenticate_user()
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    bucket.reload()
    return client, bucket


def descriptor_from_fixed_uri(
    bucket: storage.Bucket,
    uri: str,
    expected_artifact_types: set[str],
) -> dict[str, Any]:
    bucket_name, object_name = parse_gs_uri(uri)
    if bucket_name != BUCKET_NAME:
        raise AssertionError("artifact bucket mismatch")
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    artifact_type = metadata.get("artifact_type")
    file_sha = metadata.get("sha256")
    if artifact_type not in expected_artifact_types:
        raise AssertionError(
            f"{object_name}: artifact_type {artifact_type!r} is not allowed"
        )
    if not file_sha or not HEX64.fullmatch(str(file_sha)):
        raise AssertionError(f"{object_name}: remote SHA metadata missing")
    descriptor = {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": str(file_sha),
        "artifact_type": str(artifact_type),
    }
    validate_descriptor_shape(descriptor, object_name)
    return descriptor


def verify_remote_descriptor(
    bucket: storage.Bucket,
    descriptor: Mapping[str, Any],
    *,
    expected_artifact_types: set[str],
    expected_object_prefix: str | None = None,
    exact_object_name: str | None = None,
) -> tuple[storage.Blob, str]:
    validate_descriptor_shape(descriptor, "remote artifact")
    bucket_name, object_name = parse_gs_uri(str(descriptor["uri"]))
    if bucket_name != BUCKET_NAME:
        raise AssertionError("artifact bucket mismatch")
    if exact_object_name is not None and object_name != exact_object_name:
        raise AssertionError(
            f"object path mismatch: {object_name!r} != {exact_object_name!r}"
        )
    if expected_object_prefix is not None and not object_name.startswith(
        expected_object_prefix
    ):
        raise AssertionError(f"object prefix mismatch: {object_name!r}")
    if descriptor["artifact_type"] not in expected_artifact_types:
        raise AssertionError("descriptor artifact_type mismatch")
    blob = bucket.blob(object_name, generation=int(descriptor["generation"]))
    blob.reload()
    metadata = blob.metadata or {}
    if int(blob.generation) != int(descriptor["generation"]):
        raise AssertionError("remote generation mismatch")
    if int(blob.size) != int(descriptor["bytes"]):
        raise AssertionError("remote byte count mismatch")
    if metadata.get("sha256") != descriptor["sha256"]:
        raise AssertionError("remote SHA metadata mismatch")
    if metadata.get("artifact_type") != descriptor["artifact_type"]:
        raise AssertionError("remote artifact_type metadata mismatch")
    return blob, object_name


def download_descriptor(
    bucket: storage.Bucket,
    descriptor: Mapping[str, Any],
    local_path: str | Path,
    *,
    expected_artifact_types: set[str],
    expected_object_prefix: str | None = None,
    exact_object_name: str | None = None,
) -> Path:
    blob, _ = verify_remote_descriptor(
        bucket,
        descriptor,
        expected_artifact_types=expected_artifact_types,
        expected_object_prefix=expected_object_prefix,
        exact_object_name=exact_object_name,
    )
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(
        str(local_path),
        if_generation_match=int(descriptor["generation"]),
        checksum="auto",
    )
    if local_path.stat().st_size != int(descriptor["bytes"]):
        raise AssertionError("downloaded byte count mismatch")
    if sha256_file(local_path) != descriptor["sha256"]:
        raise AssertionError("downloaded SHA mismatch")
    return local_path


def load_json_descriptor(
    bucket: storage.Bucket,
    descriptor: Mapping[str, Any],
    local_name: str,
    *,
    expected_artifact_types: set[str],
    expected_object_prefix: str | None = None,
    exact_object_name: str | None = None,
) -> tuple[dict[str, Any], Path]:
    path = download_descriptor(
        bucket,
        descriptor,
        LOCAL_INPUT / local_name,
        expected_artifact_types=expected_artifact_types,
        expected_object_prefix=expected_object_prefix,
        exact_object_name=exact_object_name,
    )
    return json.loads(path.read_text(encoding="utf-8")), path


def upload_create_only(
    bucket: storage.Bucket,
    local_path: str | Path,
    object_name: str,
    artifact_type: str,
) -> dict[str, Any]:
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {"sha256": file_sha, "artifact_type": artifact_type}
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob = bucket.blob(object_name)
        blob.reload()
        metadata = blob.metadata or {}
        if (
            metadata.get("sha256") != file_sha
            or metadata.get("artifact_type") != artifact_type
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(
                f"create-only collision with different content: gs://{bucket.name}/{object_name}"
            )
    blob = bucket.blob(object_name)
    blob.reload()
    descriptor = {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    verify_remote_descriptor(
        bucket,
        descriptor,
        expected_artifact_types={artifact_type},
        exact_object_name=object_name,
    )
    return descriptor


def upload_json_create_only(
    bucket: storage.Bucket,
    payload: Mapping[str, Any],
    object_name: str,
    artifact_type: str,
) -> dict[str, Any]:
    path = LOCAL_META / (Path(object_name).name + ".local")
    path.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, path, object_name, artifact_type)


def pin_json_uri(
    bucket: storage.Bucket,
    uri: str,
    local_name: str,
    artifact_types: set[str],
) -> tuple[dict[str, Any], dict[str, Any], Path]:
    descriptor = descriptor_from_fixed_uri(bucket, uri, artifact_types)
    payload, path = load_json_descriptor(
        bucket,
        descriptor,
        local_name,
        expected_artifact_types=artifact_types,
        exact_object_name=parse_gs_uri(uri)[1],
    )
    return payload, descriptor, path


## v4 frozen router-selection contract.


In [ ]:
SELECTION_SCHEMA = "dcc-m3-v4-router-model-selection-seal-v2"
ARCHITECTURE_SCHEMA = "dcc-m3-v4-obstacle-regressor-architecture-v1"
CHECKPOINT_SCHEMA = "dcc-m3-v4-router-regressor-checkpoint-v1"
DATASET_SCHEMA = "dcc-m3-dataset-seal-v2"
EXPECTED_V4_DATASET_VERSION = "m3ds-v4-ml-fixedtest-v2"
SELECTION_ARTIFACT_TYPES = {
    "m3_v4_router_model_selection_seal",
    "m3_v4_model_selection_seal",
}
DATASET_ARTIFACT_TYPES = {"m3_v4_dataset_seal"}
REGRESSOR_CHECKPOINT_ARTIFACT_TYPES = {
    "m3_v4_router_regressor_checkpoint",
    "m3_v4_regressor_checkpoint",
}
DETECTOR_CHECKPOINT_ARTIFACT_TYPES = {
    "m3_v4_yolo26s_best_checkpoint",
}
CHECKPOINT_ARTIFACT_TYPES = (
    REGRESSOR_CHECKPOINT_ARTIFACT_TYPES | DETECTOR_CHECKPOINT_ARTIFACT_TYPES
)
EXPECTED_COMPONENT_KEYS = {
    "full_frame_regressor",
    "yolo_detector",
    "roi_regressor",
}


def normalize_type_ranges(
    value: Mapping[str, Sequence[float]],
) -> dict[str, tuple[float, float]]:
    if set(value) != set(TYPE_ORDER):
        raise AssertionError("type range key set mismatch")
    result: dict[str, tuple[float, float]] = {}
    for type_name in TYPE_ORDER:
        pair = value[type_name]
        if len(pair) != 2:
            raise AssertionError(f"invalid type range for {type_name}")
        low, high = map(float, pair)
        if not (math.isfinite(low) and math.isfinite(high) and low < high):
            raise AssertionError(f"invalid type range for {type_name}")
        result[type_name] = (low, high)
    return result


def validate_inference_rule(rule: Mapping[str, Any]) -> dict[str, Any]:
    if not isinstance(rule, Mapping):
        raise AssertionError("router inference_rule missing")
    normalized = {
        "detector_confidence": float(rule.get("detector_confidence", float("nan"))),
        "max_detections": int(rule.get("max_detections", -1)),
        "detector_imgsz": int(rule.get("detector_imgsz", -1)),
        "detector_iou": float(rule.get("detector_iou", float("nan"))),
        "crop_padding": float(rule.get("crop_padding", float("nan"))),
        "meter_fusion": rule.get("meter_fusion"),
        "obstacle_fusion": rule.get("obstacle_fusion"),
        "detector_miss_fallback": rule.get("detector_miss_fallback"),
    }
    for key, expected in EXPECTED_YOLO_INFERENCE_RULE.items():
        observed = normalized[key]
        if isinstance(expected, float):
            if not math.isclose(float(observed), expected, rel_tol=0.0, abs_tol=1e-12):
                raise AssertionError(f"router inference rule mismatch: {key}")
        elif observed != expected:
            raise AssertionError(f"router inference rule mismatch: {key}")
    invalid_policy = rule.get(
        "invalid_candidate_policy",
        "drop_invalid_or_degenerate_then_fallback_if_empty",
    )
    if invalid_policy not in {
        "drop_invalid_or_degenerate_then_fallback_if_empty",
        "drop_invalid_and_degenerate_then_fallback_if_empty",
    }:
        raise AssertionError("invalid YOLO candidate policy")
    normalized["invalid_candidate_policy"] = invalid_policy
    return normalized


def component_model_vocabulary(component: Mapping[str, Any]) -> list[str]:
    architecture = component["architecture_payload"]
    return [str(value) for value in architecture.get("obstacle_vocabulary", [])]


def validate_regressor_component(
    component: Mapping[str, Any],
    *,
    expected_component: str,
    expected_image_size: int,
) -> tuple[list[str], dict[str, tuple[float, float]]]:
    if not isinstance(component, Mapping):
        raise AssertionError(f"{expected_component} component missing")
    if component.get("component") != expected_component:
        raise AssertionError(f"{expected_component} component name mismatch")
    if not str(component.get("model_id", "")):
        raise AssertionError(f"{expected_component} model_id missing")
    if int(component.get("image_size", -1)) != expected_image_size:
        raise AssertionError(f"{expected_component} image_size mismatch")
    if not isinstance(component.get("use_obstacle_head"), bool):
        raise AssertionError(f"{expected_component} obstacle-head flag missing")

    architecture = component.get("architecture_payload")
    if not isinstance(architecture, Mapping):
        raise AssertionError(f"{expected_component} architecture missing")
    if architecture.get("schema_version") != ARCHITECTURE_SCHEMA:
        raise AssertionError(f"{expected_component} architecture schema mismatch")
    if architecture.get("model_class") != "V4ObstacleRegressor":
        raise AssertionError(f"{expected_component} model class mismatch")
    if architecture.get("forward_signature") != (
        "(images,type_index)->(pred_m,obstacle_logits_or_none)"
    ):
        raise AssertionError(f"{expected_component} forward signature mismatch")
    if canonical_sha256(architecture) != component.get("architecture_fingerprint"):
        raise AssertionError(f"{expected_component} architecture fingerprint mismatch")
    if list(architecture.get("input_contract", [])) != ["images", "type_index"]:
        raise AssertionError(f"{expected_component} input contract mismatch")
    forbidden_inputs = {
        "obstruction",
        "waterlevel_m",
        "target_m",
        "stem",
        "group_id",
        "scene_group_id",
        "residual",
    }
    if forbidden_inputs.intersection(architecture.get("input_contract", [])):
        raise AssertionError(f"{expected_component} contains target/private input")
    if architecture.get("backbone") != "convnext_tiny":
        raise AssertionError(f"{expected_component} backbone mismatch")
    if int(architecture.get("image_size", -1)) != expected_image_size:
        raise AssertionError(f"{expected_component} architecture image size mismatch")
    if int(architecture.get("type_embedding_dim", -1)) != 32:
        raise AssertionError(f"{expected_component} type embedding mismatch")
    if int(architecture.get("hidden_dim", -1)) != 256:
        raise AssertionError(f"{expected_component} hidden dimension mismatch")
    if not math.isclose(float(architecture.get("dropout", -1)), 0.15):
        raise AssertionError(f"{expected_component} dropout mismatch")
    if bool(architecture.get("use_obstacle_head")) != bool(
        component["use_obstacle_head"]
    ):
        raise AssertionError(f"{expected_component} obstacle-head mismatch")
    if list(architecture.get("type_order", [])) != TYPE_ORDER:
        raise AssertionError(f"{expected_component} type order mismatch")
    preprocessing = architecture.get("preprocessing", {})
    expected_preprocessing = {
        "resize": "letterbox",
        "fill_rgb": [114, 114, 114],
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    }
    if preprocessing != expected_preprocessing:
        raise AssertionError(f"{expected_component} preprocessing mismatch")
    type_ranges = normalize_type_ranges(
        architecture.get("type_ranges", DEFAULT_TYPE_RANGES)
    )
    if type_ranges != DEFAULT_TYPE_RANGES:
        raise AssertionError(f"{expected_component} physical ranges mismatch")

    vocabulary = component_model_vocabulary(component)
    if bool(component["use_obstacle_head"]):
        if vocabulary != OBSTACLE_ORDER:
            raise AssertionError(f"{expected_component} vocabulary mismatch")
    elif vocabulary:
        raise AssertionError(f"{expected_component} control vocabulary must be empty")

    checkpoint = component.get("checkpoint")
    if not isinstance(checkpoint, Mapping):
        raise AssertionError(f"{expected_component} checkpoint missing")
    validate_descriptor_shape(checkpoint, f"{expected_component} checkpoint")
    if checkpoint.get("artifact_type") not in REGRESSOR_CHECKPOINT_ARTIFACT_TYPES:
        raise AssertionError(f"{expected_component} checkpoint artifact mismatch")
    return vocabulary, type_ranges


def router_architecture_fingerprint(winner: Mapping[str, Any]) -> str:
    components = winner["components"]
    payload = {
        "kind": "type_router",
        "route_by_type": FIXED_ROUTE_BY_TYPE,
        "routing_sha256": winner["routing_sha256"],
        "full_frame_architecture_fingerprint": components[
            "full_frame_regressor"
        ]["architecture_fingerprint"],
        "roi_architecture_fingerprint": components[
            "roi_regressor"
        ]["architecture_fingerprint"],
        "detector_checkpoint_sha256": components["yolo_detector"]["sha256"],
    }
    return canonical_sha256(payload)


def validate_selection(
    selection: Mapping[str, Any],
) -> tuple[
    Mapping[str, Any],
    list[str],
    list[str],
    dict[str, tuple[float, float]],
]:
    if selection.get("status") != "MODEL_DEPLOYMENT_FROZEN":
        raise AssertionError("v4 router deployment is not frozen")
    if selection.get("schema_version") != SELECTION_SCHEMA:
        raise AssertionError("v4 router selection schema mismatch")
    require_all_assertions(selection, "selection seal")
    winner = selection.get("winner")
    if not isinstance(winner, Mapping):
        raise AssertionError("selection winner missing")
    if selection.get("winner_model_id") != winner.get("model_id"):
        raise AssertionError("selection winner ID mismatch")
    if selection.get("winner_kind") != "type_router" or winner.get("kind") != "type_router":
        raise AssertionError("v4 winner must be type_router")
    if selection.get("dataset_version") != EXPECTED_V4_DATASET_VERSION:
        raise AssertionError("selection dataset version mismatch")

    dataset_descriptor = selection.get("dataset_seal")
    if not isinstance(dataset_descriptor, Mapping):
        raise AssertionError("selection dataset seal descriptor missing")
    validate_descriptor_shape(dataset_descriptor, "selection dataset seal")
    if dataset_descriptor.get("artifact_type") not in DATASET_ARTIFACT_TYPES:
        raise AssertionError("selection dataset seal artifact type mismatch")
    if selection.get("dataset_seal_sha256") != dataset_descriptor["sha256"]:
        raise AssertionError("selection dataset seal SHA field mismatch")

    if dict(winner.get("route_by_type", {})) != FIXED_ROUTE_BY_TYPE:
        raise AssertionError("winner fixed type route mismatch")
    if dict(selection.get("route_by_type", FIXED_ROUTE_BY_TYPE)) != FIXED_ROUTE_BY_TYPE:
        raise AssertionError("selection fixed type route mismatch")
    routing_contract = winner.get("routing_contract")
    if not isinstance(routing_contract, Mapping):
        raise AssertionError("winner routing contract missing")
    if dict(selection.get("routing_contract", {})) != dict(routing_contract):
        raise AssertionError("selection/winner routing contract mismatch")
    if dict(routing_contract.get("route_by_type", {})) != FIXED_ROUTE_BY_TYPE:
        raise AssertionError("routing contract type route mismatch")
    if routing_contract.get("routing_key") != "type_from_sample_id_first_character":
        raise AssertionError("routing key mismatch")
    inference_rule = validate_inference_rule(routing_contract.get("inference_rule", {}))
    if dict(winner.get("inference_rule", {})) != dict(
        routing_contract.get("inference_rule", {})
    ):
        raise AssertionError("winner/routing inference rule mismatch")
    routing_sha = canonical_sha256(routing_contract)
    if winner.get("routing_sha256") != routing_sha:
        raise AssertionError("winner routing SHA mismatch")
    if selection.get("routing_sha256") != routing_sha:
        raise AssertionError("selection routing SHA mismatch")

    components = winner.get("components")
    if not isinstance(components, Mapping) or set(components) != EXPECTED_COMPONENT_KEYS:
        raise AssertionError("winner component key set mismatch")
    if selection.get("components") != components:
        raise AssertionError("selection/winner components mismatch")
    if selection.get("checkpoint_bundle_sha256") != winner.get(
        "checkpoint_bundle_sha256"
    ):
        raise AssertionError("selection/winner checkpoint bundle SHA mismatch")
    if selection.get("inference_rule") != routing_contract.get(
        "inference_rule"
    ):
        raise AssertionError("selection/routing inference rule mismatch")
    full_vocab, full_ranges = validate_regressor_component(
        components["full_frame_regressor"],
        expected_component="full_frame_regressor",
        expected_image_size=224,
    )
    roi_vocab, roi_ranges = validate_regressor_component(
        components["roi_regressor"],
        expected_component="roi_regressor",
        expected_image_size=384,
    )
    if full_ranges != roi_ranges:
        raise AssertionError("full/ROI physical range contract mismatch")
    if full_vocab != roi_vocab:
        raise AssertionError("full/ROI obstacle vocabulary mismatch")
    if bool(components["full_frame_regressor"]["use_obstacle_head"]) != bool(
        components["roi_regressor"]["use_obstacle_head"]
    ):
        raise AssertionError("full/ROI obstacle-head mode mismatch")

    detector = components["yolo_detector"]
    if not isinstance(detector, Mapping):
        raise AssertionError("YOLO detector descriptor missing")
    validate_descriptor_shape(detector, "YOLO detector checkpoint")
    if detector.get("artifact_type") not in DETECTOR_CHECKPOINT_ARTIFACT_TYPES:
        raise AssertionError("YOLO detector artifact type mismatch")
    if selection.get("detector") != detector:
        raise AssertionError("selection/winner YOLO detector mismatch")
    detector_config = selection.get("detector_config")
    if not isinstance(detector_config, Mapping):
        raise AssertionError("v4 detector training config missing")
    expected_detector_config = {
        "schema_version": "dcc-m3-v4-yolo26s-detector-config-v1",
        "mode": "retrain_v4",
        "base_weights": "yolo26s.pt",
        "ultralytics_version": "8.4.121",
        "class_names": ["mokja"],
        "epochs": 4,
        "train_cap": 60_000,
        "train_rows": 60_000,
        "train_selection": (
            "proportional_type_x_target_decile_sha256_seeded"
        ),
        "selection": (
            "proportional_type_x_target_decile_sha256_seeded"
        ),
        "imgsz": 960,
        "batch": 4,
        "conf": 0.05,
        "iou": 0.70,
        "max_det": 10,
        "seed": 42,
        "amp": True,
        "cache": False,
        "fraction": 1.0,
        "augmentation_off": [
            "mosaic",
            "mixup",
            "cutmix",
            "copy_paste",
            "multi_scale",
            "close_mosaic",
            "fliplr",
            "flipud",
            "hsv_h",
            "hsv_s",
            "hsv_v",
            "degrees",
            "translate",
            "scale",
            "shear",
            "perspective",
        ],
        "dataset_version": EXPECTED_V4_DATASET_VERSION,
    }
    for key, expected in expected_detector_config.items():
        observed = detector_config.get(key)
        if isinstance(expected, float):
            if not math.isclose(
                float(observed), expected, rel_tol=0.0, abs_tol=1e-12
            ):
                raise AssertionError(f"detector config mismatch: {key}")
        elif observed != expected:
            raise AssertionError(f"detector config mismatch: {key}")
    if int(detector_config.get("val_rows", 0)) <= 0:
        raise AssertionError("detector validation row count must be positive")
    detector_config_sha = canonical_sha256(detector_config)
    if selection.get("detector_config_sha256") != detector_config_sha:
        raise AssertionError("detector training config SHA mismatch")
    if selection.get("detector_training_contract") != detector_config:
        raise AssertionError("detector training contract/config mismatch")
    if routing_contract.get("detector_checkpoint_sha256") != detector["sha256"]:
        raise AssertionError("routing/detector checkpoint SHA mismatch")
    if routing_contract.get("detector_training_config_sha256") != (
        detector_config_sha
    ):
        raise AssertionError("routing/detector config SHA mismatch")
    if routing_contract.get("detector_frozen_between_regression_arms") is not True:
        raise AssertionError("detector must be shared and frozen between arms")
    selection_assertions = selection["assertions"]
    required_detector_assertions = {
        "shared_frozen_detector_between_arms",
        "v4_detector_retrained_default",
    }
    if required_detector_assertions - set(selection_assertions):
        raise AssertionError("detector selection assertions missing")
    if not all(
        bool(selection_assertions[key])
        for key in required_detector_assertions
    ):
        raise AssertionError("detector selection assertion failed")

    if winner.get("checkpoint_bundle_sha256") != canonical_sha256(components):
        raise AssertionError("component bundle SHA mismatch")
    computed_fingerprint = router_architecture_fingerprint(winner)
    if winner.get("architecture_fingerprint", computed_fingerprint) != computed_fingerprint:
        raise AssertionError("router architecture fingerprint mismatch")

    evaluation_vocabulary = [
        str(value) for value in selection.get("obstacle_vocabulary", [])
    ]
    if evaluation_vocabulary != OBSTACLE_ORDER:
        raise AssertionError("selection evaluation vocabulary/order mismatch")
    model_vocabulary = full_vocab
    if model_vocabulary:
        thresholds = winner.get("obstacle_thresholds")
        if not isinstance(thresholds, Sequence) or len(thresholds) != len(
            model_vocabulary
        ):
            raise AssertionError("router obstacle thresholds missing")
        if not all(
            math.isfinite(float(value)) and 0.0 <= float(value) <= 1.0
            for value in thresholds
        ):
            raise AssertionError("router obstacle thresholds invalid")

    test_contract = selection.get("test_contract")
    if not isinstance(test_contract, Mapping):
        raise AssertionError("selection public anonymous test_contract missing")
    required = {
        "inference_image_seal_sha256",
        "test_sample_id_sha256",
        "row_count",
    }
    if required - set(test_contract):
        raise AssertionError("selection test_contract is incomplete")
    if not HEX64.fullmatch(str(test_contract["inference_image_seal_sha256"])):
        raise AssertionError("invalid test-contract inference seal SHA")
    if not HEX64.fullmatch(str(test_contract["test_sample_id_sha256"])):
        raise AssertionError("invalid test-contract sample SHA")
    if int(test_contract["row_count"]) <= 0:
        raise AssertionError("invalid test-contract row count")
    return winner, model_vocabulary, evaluation_vocabulary, full_ranges


def load_and_validate_selection_dataset(
    bucket: storage.Bucket,
    selection: Mapping[str, Any],
    inference_seal: Mapping[str, Any],
    inference_descriptor: Mapping[str, Any],
    inference_seal_sha256: str,
) -> tuple[dict[str, Any], dict[str, Any], Path]:
    """Re-pin the 04A dataset seal and cross-check its fixed-test contract."""

    dataset_descriptor = selection["dataset_seal"]
    dataset, dataset_path = load_json_descriptor(
        bucket,
        dataset_descriptor,
        "pinned_v4_dataset_seal.json",
        expected_artifact_types=DATASET_ARTIFACT_TYPES,
        expected_object_prefix=(
            f"dcc/mission3/datasets/{EXPECTED_V4_DATASET_VERSION}/"
        ),
    )
    if sha256_file(dataset_path) != selection["dataset_seal_sha256"]:
        raise AssertionError("selection/dataset seal SHA mismatch")
    if dataset.get("status") != "DATASET_COMPLETE":
        raise AssertionError("v4 dataset seal is incomplete")
    if dataset.get("schema_version") != DATASET_SCHEMA:
        raise AssertionError("v4 dataset seal schema mismatch")
    if dataset.get("dataset_version") != EXPECTED_V4_DATASET_VERSION:
        raise AssertionError("v4 dataset version mismatch")
    if dataset.get("data_profile") != "full":
        raise AssertionError("v4 dataset profile must be full")
    require_all_assertions(dataset, "v4 dataset seal")

    dataset_test_contract = dataset.get("test_contract")
    selection_test_contract = selection.get("test_contract")
    if not isinstance(dataset_test_contract, Mapping):
        raise AssertionError("dataset fixed-test contract missing")
    if dict(dataset_test_contract) != dict(selection_test_contract):
        raise AssertionError("dataset/selection fixed-test contract mismatch")
    if dataset_test_contract["inference_image_seal_sha256"] != (
        inference_seal_sha256
    ):
        raise AssertionError("dataset/inference image seal SHA mismatch")
    if dataset_test_contract["inference_image_seal_sha256"] != (
        inference_descriptor["sha256"]
    ):
        raise AssertionError("dataset/inference descriptor SHA mismatch")
    if dataset_test_contract["test_sample_id_sha256"] != inference_seal.get(
        "test_sample_id_sha256"
    ):
        raise AssertionError("dataset/inference sample-set SHA mismatch")
    if int(dataset_test_contract["row_count"]) != int(
        inference_seal["counts"]["test"]
    ):
        raise AssertionError("dataset/inference test row mismatch")

    dataset_inference = dataset.get("inference")
    if not isinstance(dataset_inference, Mapping):
        raise AssertionError("dataset inference provenance missing")
    pinned_inference_descriptor = dataset_inference.get("test_image_seal")
    if not isinstance(pinned_inference_descriptor, Mapping):
        raise AssertionError("dataset inference image descriptor missing")
    validate_descriptor_shape(
        pinned_inference_descriptor, "dataset inference image seal"
    )
    descriptor_keys = sorted(DESCRIPTOR_KEYS)
    if {
        key: pinned_inference_descriptor[key] for key in descriptor_keys
    } != {key: inference_descriptor[key] for key in descriptor_keys}:
        raise AssertionError("dataset/inference descriptor provenance mismatch")
    if dataset_inference.get("test_sample_id_sha256") != (
        dataset_test_contract["test_sample_id_sha256"]
    ):
        raise AssertionError("dataset inference sample SHA field mismatch")
    if int(dataset_inference.get("test_count", -1)) != int(
        dataset_test_contract["row_count"]
    ):
        raise AssertionError("dataset inference row-count field mismatch")
    dataset_private = dataset.get("private")
    if not isinstance(dataset_private, Mapping):
        raise AssertionError("dataset private provenance missing")
    private_contract_descriptor = dataset_private.get("test_eval_contract")
    if not isinstance(private_contract_descriptor, Mapping):
        raise AssertionError("dataset private eval-contract descriptor missing")
    validate_descriptor_shape(
        private_contract_descriptor, "dataset private eval contract"
    )
    if private_contract_descriptor["artifact_type"] != (
        "m3_v4_private_test_eval_contract"
    ):
        raise AssertionError("dataset private eval-contract artifact type mismatch")
    _, private_contract_object = parse_gs_uri(
        private_contract_descriptor["uri"]
    )
    if not private_contract_object.startswith(PRIVATE_SPLIT_PREFIX):
        raise AssertionError("dataset private eval-contract prefix mismatch")
    return dataset, dict(dataset_descriptor), dataset_path


## Reference model and frozen preprocessing.


In [ ]:
class Letterbox:
    def __init__(self, size: int, fill: tuple[int, int, int] = (114, 114, 114)):
        self.size = int(size)
        self.fill = tuple(fill)

    def __call__(self, image: Image.Image) -> Image.Image:
        image = image.convert("RGB")
        width, height = image.size
        scale = min(self.size / width, self.size / height)
        new_size = (
            max(1, int(round(width * scale))),
            max(1, int(round(height * scale))),
        )
        resized = image.resize(new_size, Image.Resampling.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        offset = ((self.size - new_size[0]) // 2, (self.size - new_size[1]) // 2)
        canvas.paste(resized, offset)
        return canvas


def build_inference_transform(size: int) -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.Lambda(Letterbox(size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
        ]
    )


class V4ObstacleRegressor(nn.Module):
    """Reference architecture required by ARCHITECTURE_SCHEMA.

    Training uses these exact module names so strict state-dict loading checks
    the deployed architecture.  The regression output is bounded by the frozen
    per-type physical ranges.
    """

    def __init__(
        self,
        *,
        type_embedding_dim: int,
        hidden_dim: int,
        dropout: float,
        type_ranges: Mapping[str, tuple[float, float]],
        obstacle_count: int,
    ):
        super().__init__()
        self.backbone = models.convnext_tiny(weights=None)
        feature_dim = int(self.backbone.classifier[2].in_features)
        self.backbone.classifier[2] = nn.Identity()
        self.type_embedding = nn.Embedding(len(TYPE_ORDER), type_embedding_dim)
        self.shared_head = nn.Sequential(
            nn.Linear(feature_dim + type_embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(float(dropout)),
        )
        self.regression_head = nn.Linear(hidden_dim, 1)
        self.obstacle_head = (
            nn.Linear(hidden_dim, obstacle_count) if obstacle_count else None
        )
        self.register_buffer(
            "type_low",
            torch.tensor(
                [type_ranges[name][0] for name in TYPE_ORDER], dtype=torch.float32
            ),
        )
        self.register_buffer(
            "type_high",
            torch.tensor(
                [type_ranges[name][1] for name in TYPE_ORDER], dtype=torch.float32
            ),
        )

    def forward(
        self, images: torch.Tensor, type_index: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        image_features = self.backbone(images)
        type_features = self.type_embedding(type_index)
        shared = self.shared_head(torch.cat([image_features, type_features], dim=1))
        raw = self.regression_head(shared).squeeze(1)
        normalized = torch.sigmoid(raw)
        low = self.type_low[type_index]
        high = self.type_high[type_index]
        pred_m = low + normalized * (high - low)
        obstacle_logits = (
            self.obstacle_head(shared) if self.obstacle_head is not None else None
        )
        return pred_m, obstacle_logits


def build_model_from_component(
    component: Mapping[str, Any],
    type_ranges: Mapping[str, tuple[float, float]],
    vocabulary: Sequence[str],
) -> V4ObstacleRegressor:
    architecture = component["architecture_payload"]
    return V4ObstacleRegressor(
        type_embedding_dim=int(architecture["type_embedding_dim"]),
        hidden_dim=int(architecture["hidden_dim"]),
        dropout=float(architecture["dropout"]),
        type_ranges=type_ranges,
        obstacle_count=len(vocabulary),
    )


def load_regressor_component(
    bucket: storage.Bucket,
    component: Mapping[str, Any],
    *,
    local_name: str,
    type_ranges: Mapping[str, tuple[float, float]],
    vocabulary: Sequence[str],
    device: torch.device,
) -> tuple[V4ObstacleRegressor, Path]:
    descriptor = component["checkpoint"]
    checkpoint_path = download_descriptor(
        bucket,
        descriptor,
        LOCAL_INPUT / local_name,
        expected_artifact_types=REGRESSOR_CHECKPOINT_ARTIFACT_TYPES,
        expected_object_prefix="dcc/mission3/v4/runs/",
    )
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if checkpoint.get("schema_version") != CHECKPOINT_SCHEMA:
        raise AssertionError("v4 router regressor checkpoint schema mismatch")
    required_values = {
        "component": component["component"],
        "model_id": component["model_id"],
        "use_obstacle_head": component["use_obstacle_head"],
        "architecture_fingerprint": component["architecture_fingerprint"],
    }
    mismatches = {
        key: (checkpoint.get(key), expected)
        for key, expected in required_values.items()
        if checkpoint.get(key) != expected
    }
    if mismatches:
        raise AssertionError(f"regressor checkpoint contract mismatch: {mismatches}")
    if checkpoint.get("architecture_payload") != component["architecture_payload"]:
        raise AssertionError("regressor checkpoint architecture payload mismatch")
    if not HEX64.fullmatch(str(checkpoint.get("run_config_sha256", ""))):
        raise AssertionError("regressor checkpoint run-config SHA missing")
    if "model" not in checkpoint:
        raise AssertionError("regressor checkpoint state_dict missing")
    model = build_model_from_component(component, type_ranges, vocabulary)
    model.load_state_dict(checkpoint["model"], strict=True)
    model.to(device).eval()
    return model, checkpoint_path


def padded_crop(
    image: Image.Image,
    xyxy: Sequence[float],
    padding: float,
) -> Image.Image | None:
    if len(xyxy) != 4 or not np.isfinite(np.asarray(xyxy, dtype=float)).all():
        return None
    width, height = image.size
    x1, y1, x2, y2 = map(float, xyxy)
    if x2 <= x1 or y2 <= y1:
        return None
    x1 = max(0.0, min(float(width), x1))
    x2 = max(0.0, min(float(width), x2))
    y1 = max(0.0, min(float(height), y1))
    y2 = max(0.0, min(float(height), y2))
    box_w, box_h = x2 - x1, y2 - y1
    if box_w <= 1e-6 or box_h <= 1e-6:
        return None
    center_x, center_y = (x1 + x2) / 2.0, (y1 + y2) / 2.0
    half_w = box_w * (0.5 + float(padding))
    half_h = box_h * (0.5 + float(padding))
    px1 = int(max(0, math.floor(center_x - half_w)))
    py1 = int(max(0, math.floor(center_y - half_h)))
    px2 = int(min(width, math.ceil(center_x + half_w)))
    py2 = int(min(height, math.ceil(center_y + half_h)))
    if px2 <= px1 or py2 <= py1:
        return None
    return image.crop((px1, py1, px2, py2))


def confidence_weighted_median(
    values: Sequence[float],
    weights: Sequence[float],
) -> float:
    values_array = np.asarray(values, dtype=float)
    weights_array = np.asarray(weights, dtype=float)
    valid = (
        np.isfinite(values_array)
        & np.isfinite(weights_array)
        & (weights_array > 0.0)
    )
    if not valid.any():
        raise AssertionError("weighted median has no valid candidates")
    values_array = values_array[valid]
    weights_array = weights_array[valid]
    order = np.argsort(values_array, kind="mergesort")
    values_array = values_array[order]
    weights_array = weights_array[order]
    cumulative = np.cumsum(weights_array)
    index = int(np.searchsorted(cumulative, weights_array.sum() / 2.0, side="left"))
    return float(values_array[min(index, len(values_array) - 1)])


def confidence_weighted_mean(
    vectors: Sequence[Sequence[float]],
    weights: Sequence[float],
) -> list[float]:
    matrix = np.asarray(vectors, dtype=float)
    weights_array = np.asarray(weights, dtype=float)
    if matrix.ndim != 2 or matrix.shape[0] != len(weights_array):
        raise AssertionError("weighted obstacle mean shape mismatch")
    valid = np.isfinite(weights_array) & (weights_array > 0.0)
    if matrix.shape[1] == 0:
        return []
    valid &= np.isfinite(matrix).all(axis=1)
    if not valid.any():
        raise AssertionError("weighted obstacle mean has no valid candidates")
    result = np.average(matrix[valid], axis=0, weights=weights_array[valid])
    if not np.isfinite(result).all() or not ((result >= 0.0) & (result <= 1.0)).all():
        raise AssertionError("weighted obstacle probability is invalid")
    return [float(value) for value in result.tolist()]


## Anonymous shard validation and local materialization.


In [ ]:
SAFE_ID_PATTERN = re.compile(r"^[ABCEFGHS][0-9a-f]{24}$")
ALLOWED_META_KEYS = {"sample_id"}
FORBIDDEN_META_KEYS = {
    "stem",
    "type",
    "waterlevel_m",
    "target_m",
    "clslevel",
    "archive_key",
    "source_zip_uri",
    "group_id",
    "scene_group_id",
    "bbox",
    "obstruction",
    "annotations",
}


def write_bytes_verified(path: Path, payload: bytes) -> None:
    payload_sha = hashlib.sha256(payload).hexdigest()
    if path.exists():
        if path.stat().st_size != len(payload) or sha256_file(path) != payload_sha:
            raise RuntimeError(f"local anonymous image collision: {path}")
        return
    with open(path, "xb") as stream:
        stream.write(payload)


def materialize_anonymous_test(
    bucket: storage.Bucket,
    inference_seal: Mapping[str, Any],
) -> pd.DataFrame:
    if inference_seal.get("status") != "INFERENCE_IMAGES_COMPLETE":
        raise AssertionError("inference image seal status mismatch")
    if inference_seal.get("schema_version") != "dcc-m3-inference-image-seal-v1":
        raise AssertionError("inference image seal schema mismatch")
    if inference_seal.get("dataset_version") != EXPECTED_INFERENCE_DATASET_VERSION:
        raise AssertionError("unexpected anonymous inference dataset")
    require_all_assertions(inference_seal, "inference image seal")
    if inference_seal.get("allowed_metadata") != ["sample_id"]:
        raise AssertionError("anonymous metadata allowlist mismatch")
    # The pinned v1 seal predates these two denylist names.  This is only
    # a declaration check; actual shard metadata still permits sample_id only
    # and is checked against the complete FORBIDDEN_META_KEYS set below.
    if not FORBIDDEN_META_KEYS.difference({"scene_group_id", "annotations"}).issubset(
        set(inference_seal.get("forbidden_metadata", []))
    ):
        raise AssertionError("anonymous metadata denylist is incomplete")
    expected_rows = int(inference_seal["counts"]["test"])
    if expected_rows <= 0:
        raise AssertionError("anonymous test count is invalid")
    descriptors = list(inference_seal.get("test_shards", []))
    if not descriptors:
        raise AssertionError("anonymous test shards missing")
    required_bytes = sum(int(item["bytes"]) for item in descriptors)
    if shutil.disk_usage("/content").free < required_bytes + 12 * 1024**3:
        raise RuntimeError("insufficient local disk for anonymous test shards")

    prefix = f"dcc/inference/mission3/{EXPECTED_INFERENCE_DATASET_VERSION}/test_images/"
    metadata_rows: list[dict[str, str]] = []
    seen_ids: set[str] = set()
    for shard_index, descriptor in enumerate(descriptors):
        if "archive_key" in descriptor:
            raise AssertionError("anonymous shard descriptor exposes archive_key")
        local_shard = LOCAL_SHARDS / (
            f"{str(descriptor['sha256'])[:16]}-test-{shard_index:04d}.tar"
        )
        download_descriptor(
            bucket,
            descriptor,
            local_shard,
            expected_artifact_types={"m3_test_image_shard"},
            expected_object_prefix=prefix,
        )
        with tarfile.open(local_shard, "r:") as handle:
            regular_members = [member for member in handle.getmembers() if member.isfile()]
            member_by_name = {member.name: member for member in regular_members}
            if len(member_by_name) != len(regular_members):
                raise AssertionError("duplicate tar member name")
            if any("/" in name or "\\" in name for name in member_by_name):
                raise AssertionError("anonymous tar contains nested paths")
            keys = sorted({Path(name).stem for name in member_by_name})
            for key in keys:
                if not SAFE_ID_PATTERN.fullmatch(key):
                    raise AssertionError(f"invalid anonymous sample_id: {key}")
                if key in seen_ids:
                    raise AssertionError(f"duplicate anonymous sample_id: {key}")
                expected_pair = {f"{key}.jpg", f"{key}.json"}
                actual_pair = {name for name in member_by_name if Path(name).stem == key}
                if actual_pair != expected_pair:
                    raise AssertionError(f"anonymous tar pair mismatch: {key}")
                raw_meta = handle.extractfile(member_by_name[f"{key}.json"]).read()
                metadata = json.loads(raw_meta.decode("utf-8"))
                if set(metadata) != ALLOWED_META_KEYS:
                    raise AssertionError("anonymous metadata key contract failed")
                if FORBIDDEN_META_KEYS.intersection(metadata):
                    raise AssertionError("private field found in anonymous metadata")
                if metadata["sample_id"] != key:
                    raise AssertionError("anonymous metadata sample_id mismatch")
                image_bytes = handle.extractfile(member_by_name[f"{key}.jpg"]).read()
                write_bytes_verified(LOCAL_IMAGES / f"{key}.jpg", image_bytes)
                seen_ids.add(key)
                metadata_rows.append({"sample_id": key, "type": key[0]})

    frame = pd.DataFrame(metadata_rows).sort_values(
        "sample_id", kind="mergesort"
    ).reset_index(drop=True)
    if len(frame) != expected_rows or not frame["sample_id"].is_unique:
        raise AssertionError("anonymous test row contract failed")
    if not frame["type"].isin(TYPE_ORDER).all():
        raise AssertionError("anonymous type contract failed")
    observed_sha = canonical_sha256(sorted(frame["sample_id"].tolist()))
    if observed_sha != inference_seal["test_sample_id_sha256"]:
        raise AssertionError("anonymous sample_id SHA mismatch")
    return frame


class AnonymousImageDataset(Dataset):
    def __init__(self, metadata: pd.DataFrame, transform: Any):
        self.rows = metadata[["sample_id", "type"]].to_dict(orient="records")
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, str, str]:
        row = self.rows[index]
        sample_id = str(row["sample_id"])
        type_name = str(row["type"])
        if type_name != sample_id[0]:
            raise AssertionError("dataset type/sample_id mismatch")
        with Image.open(LOCAL_IMAGES / f"{sample_id}.jpg") as source:
            image = source.convert("RGB")
        return (
            self.transform(image),
            torch.tensor(TYPE_TO_INDEX[type_name], dtype=torch.long),
            sample_id,
            type_name,
        )


## INFERENCE phase: frozen model, anonymous images, prediction seal.


In [ ]:
def run_inference() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("INFERENCE phase requires a Colab GPU runtime")
    _, bucket = authenticate_storage()

    selection, selection_descriptor, selection_path = pin_json_uri(
        bucket,
        SELECTION_SEAL_URI,
        "v4_router_model_selection_seal.json",
        SELECTION_ARTIFACT_TYPES,
    )
    (
        winner,
        model_vocabulary,
        evaluation_vocabulary,
        type_ranges,
    ) = validate_selection(selection)
    components = winner["components"]
    route_by_type = dict(winner["route_by_type"])
    routing_contract = dict(winner["routing_contract"])
    inference_rule = validate_inference_rule(routing_contract["inference_rule"])
    router_fingerprint = router_architecture_fingerprint(winner)

    inference_seal, inference_descriptor, inference_seal_path = pin_json_uri(
        bucket,
        INFERENCE_IMAGE_SEAL_URI,
        "existing_anonymous_test_image_seal.json",
        {"m3_inference_image_seal"},
    )
    if inference_seal.get("dataset_version") != EXPECTED_INFERENCE_DATASET_VERSION:
        raise AssertionError("wrong anonymous inference seal")
    require_all_assertions(inference_seal, "inference image seal")
    inference_seal_sha = sha256_file(inference_seal_path)
    test_contract = selection["test_contract"]
    if test_contract["inference_image_seal_sha256"] != inference_seal_sha:
        raise AssertionError("selection and inference-image seal SHA mismatch")
    if test_contract["test_sample_id_sha256"] != inference_seal["test_sample_id_sha256"]:
        raise AssertionError("selection and inference sample-set SHA mismatch")
    if int(test_contract["row_count"]) != int(inference_seal["counts"]["test"]):
        raise AssertionError("selection and inference test row mismatch")
    dataset_descriptor = dict(selection["dataset_seal"])
    dataset_seal_sha = str(selection["dataset_seal_sha256"])
    dataset_version = str(selection["dataset_version"])

    metadata = materialize_anonymous_test(bucket, inference_seal)
    observed_sample_sha = canonical_sha256(sorted(metadata["sample_id"].tolist()))
    if observed_sample_sha != test_contract["test_sample_id_sha256"]:
        raise AssertionError("materialized sample set differs from selection")
    observed_route_counts = (
        metadata["type"].map(route_by_type).value_counts().sort_index().to_dict()
    )
    if observed_route_counts != EXPECTED_ROUTE_COUNTS:
        raise AssertionError(
            f"fixed anonymous route count mismatch: {observed_route_counts}"
        )

    device = torch.device("cuda")
    full_component = components["full_frame_regressor"]
    roi_component = components["roi_regressor"]
    full_model, _ = load_regressor_component(
        bucket,
        full_component,
        local_name="v4_full_frame_regressor.pt",
        type_ranges=type_ranges,
        vocabulary=model_vocabulary,
        device=device,
    )
    full_transform = build_inference_transform(int(full_component["image_size"]))
    full_loader = DataLoader(
        AnonymousImageDataset(metadata, full_transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    full_predictions: dict[str, float] = {}
    full_probabilities: dict[str, list[float]] = {}
    with torch.inference_mode():
        for images, type_index, sample_ids, _ in full_loader:
            images = images.to(device, non_blocking=True)
            type_index = type_index.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=True):
                pred_m, obstacle_logits = full_model(images, type_index)
            pred_values = pred_m.float().cpu().numpy()
            probability_values = (
                torch.sigmoid(obstacle_logits).float().cpu().numpy()
                if obstacle_logits is not None
                else None
            )
            for offset, (sample_id, value) in enumerate(zip(sample_ids, pred_values)):
                sample_id = str(sample_id)
                if sample_id in full_predictions:
                    raise AssertionError("duplicate full-frame prediction")
                full_predictions[sample_id] = float(value)
                full_probabilities[sample_id] = (
                    []
                    if probability_values is None
                    else [
                        float(probability)
                        for probability in probability_values[offset].tolist()
                    ]
                )
    if set(full_predictions) != set(metadata["sample_id"]):
        raise AssertionError("full-frame prediction completeness failed")
    if set(full_probabilities) != set(full_predictions):
        raise AssertionError("full-frame obstacle prediction completeness failed")
    del full_model, full_loader
    gc.collect()
    torch.cuda.empty_cache()

    detector_descriptor = components["yolo_detector"]
    detector_path = download_descriptor(
        bucket,
        detector_descriptor,
        LOCAL_INPUT / "v4_yolo26s_best.pt",
        expected_artifact_types=DETECTOR_CHECKPOINT_ARTIFACT_TYPES,
        expected_object_prefix="dcc/mission3/v4/runs/",
    )
    roi_model, _ = load_regressor_component(
        bucket,
        roi_component,
        local_name="v4_roi_regressor.pt",
        type_ranges=type_ranges,
        vocabulary=model_vocabulary,
        device=device,
    )
    roi_transform = build_inference_transform(int(roi_component["image_size"]))

    detector_ids = set(
        metadata.loc[
            metadata["type"].map(route_by_type).eq("yolo_roi"), "sample_id"
        ].astype(str)
    )
    if len(detector_ids) != EXPECTED_ROUTE_COUNTS["yolo_roi"]:
        raise AssertionError("YOLO routed ID count mismatch")
    for sample_id in sorted(detector_ids):
        source = LOCAL_IMAGES / f"{sample_id}.jpg"
        target = LOCAL_ROUTED_IMAGES / source.name
        if target.exists():
            if target.stat().st_size != source.stat().st_size:
                raise AssertionError("stale routed detector image")
        else:
            try:
                os.link(source, target)
            except OSError:
                shutil.copyfile(source, target)
    observed_detector_ids = {path.stem for path in LOCAL_ROUTED_IMAGES.glob("*.jpg")}
    if observed_detector_ids != detector_ids:
        raise AssertionError("routed detector directory ID set mismatch")

    detector = YOLO(str(detector_path))
    if isinstance(getattr(detector, "ckpt", None), dict):
        detector.ckpt["optimizer"] = None
        detector.ckpt["scaler"] = None

    crop_outputs: dict[str, list[tuple[float, list[float], float]]] = {
        sample_id: [] for sample_id in detector_ids
    }
    detection_summary: dict[str, dict[str, Any]] = {}
    crop_tensors: list[torch.Tensor] = []
    crop_type_indices: list[int] = []
    crop_sample_ids: list[str] = []
    crop_weights: list[float] = []

    def flush_crops() -> None:
        if not crop_tensors:
            return
        images = torch.stack(crop_tensors).to(device, non_blocking=True)
        type_index = torch.tensor(
            crop_type_indices, dtype=torch.long, device=device
        )
        with torch.inference_mode(), torch.amp.autocast(
            device_type="cuda", enabled=True
        ):
            pred_m, obstacle_logits = roi_model(images, type_index)
        pred_values = pred_m.float().cpu().numpy()
        probability_values = (
            torch.sigmoid(obstacle_logits).float().cpu().numpy()
            if obstacle_logits is not None
            else None
        )
        for offset, (sample_id, weight, value) in enumerate(
            zip(crop_sample_ids, crop_weights, pred_values)
        ):
            probabilities = (
                []
                if probability_values is None
                else [
                    float(probability)
                    for probability in probability_values[offset].tolist()
                ]
            )
            crop_outputs[sample_id].append(
                (float(value), probabilities, float(weight))
            )
        crop_tensors.clear()
        crop_type_indices.clear()
        crop_sample_ids.clear()
        crop_weights.clear()

    detector_stream = detector.predict(
        source=str(LOCAL_ROUTED_IMAGES),
        imgsz=int(inference_rule["detector_imgsz"]),
        batch=DETECTOR_BATCH_SIZE,
        conf=float(inference_rule["detector_confidence"]),
        iou=float(inference_rule["detector_iou"]),
        max_det=int(inference_rule["max_detections"]),
        device=0,
        stream=True,
        save=False,
        verbose=False,
    )
    for result_index, result in enumerate(detector_stream, start=1):
        sample_id = Path(result.path).stem
        if sample_id not in detector_ids or sample_id in detection_summary:
            raise AssertionError("detector returned an unexpected or duplicate sample")
        type_name = sample_id[0]
        raw_boxes = (
            result.boxes.xyxy.detach().cpu().numpy()
            if result.boxes is not None
            else np.empty((0, 4), dtype=float)
        )
        raw_confidences = (
            result.boxes.conf.detach().cpu().numpy()
            if result.boxes is not None
            else np.empty(0, dtype=float)
        )
        if len(raw_boxes) != len(raw_confidences):
            raise AssertionError("detector box/confidence count mismatch")
        valid_boxes: list[list[float]] = []
        valid_confidences: list[float] = []
        with Image.open(LOCAL_IMAGES / f"{sample_id}.jpg") as source:
            image = source.convert("RGB")
        for raw_box, raw_confidence in zip(raw_boxes, raw_confidences):
            confidence = float(raw_confidence)
            if not math.isfinite(confidence) or not (0.0 <= confidence <= 1.0):
                continue
            crop = padded_crop(
                image,
                raw_box,
                float(inference_rule["crop_padding"]),
            )
            if crop is None:
                continue
            box = [float(value) for value in np.asarray(raw_box, dtype=float)]
            valid_boxes.append(box)
            valid_confidences.append(confidence)
            crop_tensors.append(roi_transform(crop))
            crop_type_indices.append(TYPE_TO_INDEX[type_name])
            crop_sample_ids.append(sample_id)
            crop_weights.append(max(confidence, 1e-6))
            if len(crop_tensors) >= BATCH_SIZE:
                flush_crops()
        detection_summary[sample_id] = {
            "detector_conf": (
                float(max(valid_confidences))
                if valid_confidences
                else float("nan")
            ),
            "n_candidates": int(len(valid_boxes)),
            "n_raw_candidates": int(len(raw_boxes)),
            "n_invalid_candidates": int(len(raw_boxes) - len(valid_boxes)),
            "candidate_boxes_xyxy_json": json.dumps(
                np.asarray(valid_boxes, dtype=float).round(3).tolist(),
                separators=(",", ":"),
                allow_nan=False,
            ),
            "candidate_confidences_json": json.dumps(
                np.asarray(valid_confidences, dtype=float).round(6).tolist(),
                separators=(",", ":"),
                allow_nan=False,
            ),
        }
        if result_index % 2_000 == 0:
            print("test detector:", result_index, "/", len(detector_ids))
    flush_crops()
    if set(detection_summary) != detector_ids:
        raise AssertionError("detector routed sample completeness failed")

    prediction_rows: list[dict[str, Any]] = []
    diagnostic_rows: list[dict[str, Any]] = []
    for row in metadata.itertuples(index=False):
        sample_id = str(row.sample_id)
        type_name = str(row.type)
        route_component = route_by_type[type_name]
        if route_component == "full_frame":
            pred_m = full_predictions[sample_id]
            probabilities = full_probabilities[sample_id]
            diagnostic = {
                "detector_conf": float("nan"),
                "n_candidates": 0,
                "n_raw_candidates": 0,
                "n_invalid_candidates": 0,
                "fallback_used": False,
                "candidate_boxes_xyxy_json": "[]",
                "candidate_confidences_json": "[]",
            }
        else:
            candidates = crop_outputs[sample_id]
            fallback_used = len(candidates) == 0
            if fallback_used:
                pred_m = full_predictions[sample_id]
                probabilities = full_probabilities[sample_id]
            else:
                values = [item[0] for item in candidates]
                probability_vectors = [item[1] for item in candidates]
                weights = [item[2] for item in candidates]
                pred_m = confidence_weighted_median(values, weights)
                probabilities = confidence_weighted_mean(
                    probability_vectors, weights
                )
            diagnostic = {
                **detection_summary[sample_id],
                "fallback_used": fallback_used,
            }
        prediction_rows.append(
            {
                "sample_id": sample_id,
                "type": type_name,
                "pred_m": float(pred_m),
            }
        )
        diagnostic_rows.append(
            {
                "sample_id": sample_id,
                "route_component": route_component,
                **diagnostic,
                "obstacle_probabilities_json": json.dumps(
                    probabilities, separators=(",", ":"), allow_nan=False
                ),
            }
        )

    predictions = pd.DataFrame(prediction_rows).sort_values(
        "sample_id", kind="mergesort"
    ).reset_index(drop=True)
    expected_rows = int(test_contract["row_count"])
    expected_diagnostic_columns = [
        "sample_id",
        "route_component",
        "detector_conf",
        "n_candidates",
        "n_raw_candidates",
        "n_invalid_candidates",
        "fallback_used",
        "candidate_boxes_xyxy_json",
        "candidate_confidences_json",
        "obstacle_probabilities_json",
    ]
    diagnostics = pd.DataFrame(diagnostic_rows).sort_values(
        "sample_id", kind="mergesort"
    ).reset_index(drop=True)
    if set(diagnostics.columns) != set(expected_diagnostic_columns):
        raise AssertionError("diagnostic schema mismatch")
    # Pandas preserves the key order of the first record.  The first sorted
    # sample uses the YOLO route, whose diagnostic mapping places
    # fallback_used after the candidate fields.  Canonicalize the serialized
    # column order after verifying that no field is missing or unexpected.
    diagnostics = diagnostics.loc[:, expected_diagnostic_columns]
    if list(predictions.columns) != ["sample_id", "type", "pred_m"]:
        raise AssertionError("prediction schema mismatch")
    if list(diagnostics.columns) != expected_diagnostic_columns:
        raise AssertionError("diagnostic schema mismatch")
    if len(predictions) != expected_rows or len(diagnostics) != expected_rows:
        raise AssertionError("prediction row count mismatch")
    if not predictions["sample_id"].is_unique or not diagnostics["sample_id"].is_unique:
        raise AssertionError("prediction sample_id duplicate")
    if predictions["sample_id"].tolist() != metadata["sample_id"].tolist():
        raise AssertionError("prediction sample order mismatch")
    if predictions["sample_id"].tolist() != diagnostics["sample_id"].tolist():
        raise AssertionError("prediction/diagnostic ID mismatch")
    if not np.isfinite(predictions["pred_m"].to_numpy(float)).all():
        raise AssertionError("non-finite regression prediction")
    if predictions["type"].tolist() != predictions["sample_id"].str[0].tolist():
        raise AssertionError("prediction type provenance mismatch")
    for type_name, (low, high) in type_ranges.items():
        values = predictions.loc[predictions["type"].eq(type_name), "pred_m"]
        if values.empty or not values.between(low - 1e-6, high + 1e-6).all():
            raise AssertionError(f"prediction range contract failed: {type_name}")

    expected_routes = predictions["type"].map(route_by_type)
    if diagnostics["route_component"].tolist() != expected_routes.tolist():
        raise AssertionError("diagnostic route differs from frozen router")
    if diagnostics["route_component"].value_counts().to_dict() != EXPECTED_ROUTE_COUNTS:
        raise AssertionError("diagnostic route count mismatch")
    full_mask = diagnostics["route_component"].eq("full_frame")
    if diagnostics.loc[full_mask, "fallback_used"].astype(bool).any():
        raise AssertionError("full-frame route cannot be marked as fallback")
    if not diagnostics.loc[full_mask, "candidate_boxes_xyxy_json"].eq("[]").all():
        raise AssertionError("full-frame route must have empty candidate boxes")
    if not diagnostics.loc[full_mask, "candidate_confidences_json"].eq("[]").all():
        raise AssertionError("full-frame route must have empty candidate confidences")
    if not diagnostics.loc[full_mask, "n_candidates"].eq(0).all():
        raise AssertionError("full-frame route must have zero candidates")
    if diagnostics.loc[~full_mask, "n_candidates"].lt(0).any():
        raise AssertionError("YOLO candidate count is negative")
    if not (
        diagnostics["n_raw_candidates"]
        == diagnostics["n_candidates"] + diagnostics["n_invalid_candidates"]
    ).all():
        raise AssertionError("raw/valid/invalid candidate identity failed")

    parsed_probabilities = diagnostics["obstacle_probabilities_json"].map(json.loads)
    expected_probability_count = len(model_vocabulary)
    if not parsed_probabilities.map(len).eq(expected_probability_count).all():
        raise AssertionError("obstacle probability vector length mismatch")
    if expected_probability_count:
        probability_matrix = np.asarray(parsed_probabilities.tolist(), dtype=float)
        if not np.isfinite(probability_matrix).all():
            raise AssertionError("non-finite obstacle probability")
        if not ((probability_matrix >= 0.0) & (probability_matrix <= 1.0)).all():
            raise AssertionError("obstacle probability outside [0,1]")

    component_checkpoints = {
        "full_frame_regressor": full_component["checkpoint"],
        "yolo_detector": detector_descriptor,
        "roi_regressor": roi_component["checkpoint"],
    }
    inference_contract = {
        "schema_version": "dcc-m3-v4-router-test-inference-contract-v2",
        "prediction_run_id": PREDICTION_RUN_ID,
        "selection_seal": selection_descriptor,
        "selection_seal_sha256": sha256_file(selection_path),
        "dataset_seal": dataset_descriptor,
        "dataset_seal_sha256": dataset_seal_sha,
        "dataset_version": dataset_version,
        "inference_image_seal": inference_descriptor,
        "inference_image_seal_sha256": inference_seal_sha,
        "anonymous_dataset_version": inference_seal["dataset_version"],
        "test_sample_id_sha256": observed_sample_sha,
        "winner_model_id": winner["model_id"],
        "winner_kind": "type_router",
        "architecture_fingerprint": router_fingerprint,
        "components": components,
        "component_checkpoints": component_checkpoints,
        "checkpoint_bundle_sha256": winner["checkpoint_bundle_sha256"],
        "routing_contract": routing_contract,
        "routing_sha256": winner["routing_sha256"],
        "input_contract": ["images", "type_index"],
        "model_obstacle_vocabulary_sha256": canonical_sha256(model_vocabulary),
        "evaluation_obstacle_vocabulary_sha256": canonical_sha256(
            evaluation_vocabulary
        ),
    }
    inference_contract_sha = canonical_sha256(inference_contract)
    if TEST_USE_POLICY == "strict_single_use":
        lease_object = (
            "dcc/inference/mission3/v4/test_use_leases/strict/"
            f"sample-set-{observed_sample_sha}.json"
        )
    else:
        lease_object = (
            "dcc/inference/mission3/v4/test_use_leases/research/"
            f"sample-set-{observed_sample_sha}/{PREDICTION_RUN_ID}.json"
        )
    test_use_lease = {
        "status": "TEST_USE_RECORDED",
        "schema_version": "dcc-m3-v4-router-test-use-lease-v2",
        "test_use_policy": TEST_USE_POLICY,
        "claim_scope": CLAIM_SCOPE,
        "prediction_run_id": PREDICTION_RUN_ID,
        "test_sample_id_sha256": observed_sample_sha,
        "selection_seal_sha256": inference_contract["selection_seal_sha256"],
        "winner_model_id": winner["model_id"],
        "checkpoint_bundle_sha256": winner["checkpoint_bundle_sha256"],
        "routing_sha256": winner["routing_sha256"],
        "inference_contract_sha256": inference_contract_sha,
    }
    lease_descriptor = upload_json_create_only(
        bucket, test_use_lease, lease_object, "m3_v4_router_test_use_lease"
    )

    attempt_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-"
        + os.urandom(4).hex()
    )
    attempt_prefix = f"{PREDICTION_OUTPUT_PREFIX}/attempts/{attempt_id}"
    prediction_path = LOCAL_OUTPUT / "test_predictions.csv"
    diagnostics_path = LOCAL_OUTPUT / "test_diagnostics.parquet"
    predictions.to_csv(
        prediction_path,
        index=False,
        encoding="utf-8",
        float_format="%.17g",
        lineterminator="\n",
    )
    diagnostics.to_parquet(diagnostics_path, index=False, compression="zstd")
    prediction_descriptor = upload_create_only(
        bucket,
        prediction_path,
        f"{attempt_prefix}/test_predictions.csv",
        "m3_v4_router_anonymous_test_predictions",
    )
    diagnostics_descriptor = upload_create_only(
        bucket,
        diagnostics_path,
        f"{attempt_prefix}/test_diagnostics.parquet",
        "m3_v4_router_anonymous_test_diagnostics",
    )

    prediction_assertions = {
        "row_count_exact": len(predictions) == expected_rows,
        "sample_ids_unique": bool(predictions["sample_id"].is_unique),
        "sample_id_set_exact": canonical_sha256(
            sorted(predictions["sample_id"].tolist())
        )
        == observed_sample_sha,
        "predictions_finite": bool(
            np.isfinite(predictions["pred_m"].to_numpy(float)).all()
        ),
        "type_from_sample_id_only": predictions["type"].tolist()
        == predictions["sample_id"].str[0].tolist(),
        "route_exact": diagnostics["route_component"].tolist()
        == expected_routes.tolist(),
        "route_counts_exact": diagnostics["route_component"].value_counts().to_dict()
        == EXPECTED_ROUTE_COUNTS,
        "full_route_candidate_arrays_empty": bool(
            diagnostics.loc[full_mask, "candidate_boxes_xyxy_json"].eq("[]").all()
            and diagnostics.loc[
                full_mask, "candidate_confidences_json"
            ].eq("[]").all()
        ),
        "full_route_fallback_false": bool(
            ~diagnostics.loc[full_mask, "fallback_used"].astype(bool).any()
        ),
        "fallback_uses_same_full_predictions": all(
            math.isclose(
                float(predictions.loc[index, "pred_m"]),
                full_predictions[str(predictions.loc[index, "sample_id"])],
                rel_tol=0.0,
                abs_tol=1e-12,
            )
            for index in diagnostics.index[
                diagnostics["fallback_used"].astype(bool)
            ]
        ),
        "prediction_ranges_valid": all(
            predictions.loc[predictions["type"].eq(type_name), "pred_m"]
            .between(low - 1e-6, high + 1e-6)
            .all()
            for type_name, (low, high) in type_ranges.items()
        ),
        "component_bundle_reverified": canonical_sha256(components)
        == winner["checkpoint_bundle_sha256"],
        "routing_contract_reverified": canonical_sha256(routing_contract)
        == winner["routing_sha256"],
        "obstacle_probabilities_valid": bool(
            parsed_probabilities.map(len).eq(expected_probability_count).all()
        ),
        "selection_test_contract_crosschecked": (
            test_contract["inference_image_seal_sha256"] == inference_seal_sha
            and test_contract["test_sample_id_sha256"] == observed_sample_sha
            and int(test_contract["row_count"]) == expected_rows
        ),
        "truth_not_accessed": True,
        "private_mapping_not_accessed": True,
        "private_contract_not_loaded": True,
    }
    failed = [key for key, value in prediction_assertions.items() if not bool(value)]
    if failed:
        raise AssertionError(f"prediction hard gate failed: {failed}")

    prediction_seal = {
        "status": "TEST_PREDICTIONS_FROZEN",
        "schema_version": "dcc-m3-v4-router-prediction-seal-v2",
        "prediction_run_id": PREDICTION_RUN_ID,
        "attempt_id": attempt_id,
        "attempt_prefix": f"gs://{BUCKET_NAME}/{attempt_prefix}",
        "claim_scope": CLAIM_SCOPE,
        "test_use_policy": TEST_USE_POLICY,
        "selection_id": selection["selection_id"],
        "selection_seal": selection_descriptor,
        "selection_seal_sha256": sha256_file(selection_path),
        "dataset_seal": dataset_descriptor,
        "dataset_seal_sha256": dataset_seal_sha,
        "dataset_version": dataset_version,
        "winner_model_id": winner["model_id"],
        "winner_kind": "type_router",
        "architecture_fingerprint": router_fingerprint,
        "components": components,
        "component_checkpoints": component_checkpoints,
        "checkpoint_bundle_sha256": winner["checkpoint_bundle_sha256"],
        "routing_contract": routing_contract,
        "routing_sha256": winner["routing_sha256"],
        "route_by_type": route_by_type,
        "route_counts": EXPECTED_ROUTE_COUNTS,
        "inference_image_seal": inference_descriptor,
        "inference_image_seal_sha256": inference_seal_sha,
        "anonymous_dataset_version": inference_seal["dataset_version"],
        "row_count": expected_rows,
        "columns": ["sample_id", "type", "pred_m"],
        "diagnostics_columns": diagnostics.columns.tolist(),
        "test_sample_id_sha256": observed_sample_sha,
        "type_counts": {
            str(key): int(value)
            for key, value in predictions["type"].value_counts().sort_index().items()
        },
        "model_obstacle_vocabulary": model_vocabulary,
        "model_obstacle_vocabulary_sha256": canonical_sha256(model_vocabulary),
        "obstacle_vocabulary": evaluation_vocabulary,
        "obstacle_vocabulary_sha256": canonical_sha256(
            evaluation_vocabulary
        ),
        "prediction": prediction_descriptor,
        "diagnostics": diagnostics_descriptor,
        "test_use_lease": lease_descriptor,
        "inference_contract": inference_contract,
        "inference_contract_sha256": inference_contract_sha,
        "inference_environment": {
            "python": platform.python_version(),
            "torch": torch.__version__,
            "torchvision": torchvision.__version__,
            "ultralytics": ultralytics.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
        },
        "truth_accessed": False,
        "assertions": prediction_assertions,
    }
    seal_descriptor = upload_json_create_only(
        bucket,
        prediction_seal,
        f"{PREDICTION_OUTPUT_PREFIX}/prediction_seal.json",
        "m3_v4_router_prediction_seal",
    )
    print("TEST PREDICTIONS FROZEN")
    print("prediction:", prediction_descriptor["uri"])
    print("prediction seal:", seal_descriptor["uri"])
    print("route counts:", EXPECTED_ROUTE_COUNTS)
    print(
        "fallback rate:",
        float(
            diagnostics.loc[
                diagnostics["route_component"].eq("yolo_roi"),
                "fallback_used",
            ].mean()
        ),
    )
    print("truth accessed: False")
    print("claim scope:", CLAIM_SCOPE)


## EVALUATION helpers: prediction-chain verification before private data.


In [ ]:
def load_private_eval_contract(
    bucket: storage.Bucket,
) -> tuple[dict[str, Any], dict[str, Any], Path]:
    contract, contract_descriptor, contract_path = pin_json_uri(
        bucket,
        PRIVATE_TEST_EVAL_CONTRACT_URI,
        "pinned_v4_private_test_eval_contract.json",
        {"m3_v4_private_test_eval_contract"},
    )
    _, contract_object = parse_gs_uri(PRIVATE_TEST_EVAL_CONTRACT_URI)
    if not contract_object.startswith(PRIVATE_SPLIT_PREFIX):
        raise AssertionError("private evaluation contract prefix mismatch")
    if contract.get("schema_version") != (
        "dcc-m3-v4-private-test-eval-contract-v2"
    ):
        raise AssertionError("private test-evaluation contract schema mismatch")
    if contract.get("claim_scope") != CLAIM_SCOPE:
        raise AssertionError("private evaluation claim scope mismatch")
    if int(contract.get("row_count", -1)) <= 0:
        raise AssertionError("private evaluation row count invalid")
    if not HEX64.fullmatch(str(contract.get("test_sample_id_sha256", ""))):
        raise AssertionError("private evaluation sample SHA invalid")
    columns = [str(value) for value in contract.get("columns", [])]
    if columns != PRIVATE_REQUIRED_COLUMNS:
        raise AssertionError("private evaluation column contract mismatch")
    vocabulary = [str(value) for value in contract.get("obstacle_vocabulary", [])]
    if vocabulary != OBSTACLE_ORDER:
        raise AssertionError("private evaluation vocabulary/order mismatch")
    if contract.get("obstacle_vocabulary_sha256") != canonical_sha256(vocabulary):
        raise AssertionError("private evaluation vocabulary SHA mismatch")
    descriptor = contract.get("test_eval_metadata")
    if not isinstance(descriptor, Mapping):
        raise AssertionError("private test_eval_metadata descriptor missing")
    validate_descriptor_shape(descriptor, "private test_eval_metadata")
    if descriptor["artifact_type"] != "m3_v4_private_test_eval_metadata":
        raise AssertionError("private test_eval_metadata artifact type mismatch")
    _, metadata_object = parse_gs_uri(descriptor["uri"])
    if not metadata_object.startswith(PRIVATE_SPLIT_PREFIX):
        raise AssertionError("private test_eval_metadata prefix mismatch")
    if "rows" in descriptor and int(descriptor["rows"]) != int(
        contract["row_count"]
    ):
        raise AssertionError("private metadata descriptor row count mismatch")
    return contract, contract_descriptor, contract_path


def verify_prediction_chain_before_truth(
    bucket: storage.Bucket,
) -> tuple[
    dict[str, Any],
    dict[str, Any],
    dict[str, Any],
    dict[str, Any],
    pd.DataFrame,
    pd.DataFrame,
    dict[str, Any],
    Path,
]:
    prediction_seal, prediction_seal_descriptor, prediction_seal_path = pin_json_uri(
        bucket,
        PREDICTION_SEAL_URI,
        "v4_router_prediction_seal.json",
        {"m3_v4_router_prediction_seal"},
    )
    if prediction_seal.get("status") != "TEST_PREDICTIONS_FROZEN":
        raise AssertionError("prediction seal is not frozen")
    if prediction_seal.get("schema_version") != (
        "dcc-m3-v4-router-prediction-seal-v2"
    ):
        raise AssertionError("router prediction seal schema mismatch")
    if prediction_seal.get("truth_accessed") is not False:
        raise AssertionError("prediction runtime accessed truth")
    require_all_assertions(prediction_seal, "prediction seal")
    if prediction_seal.get("claim_scope") != CLAIM_SCOPE:
        raise AssertionError("prediction claim scope mismatch")
    if prediction_seal.get("test_use_policy") != TEST_USE_POLICY:
        raise AssertionError("prediction test-use policy mismatch")

    selection_descriptor = prediction_seal["selection_seal"]
    selection, selection_path = load_json_descriptor(
        bucket,
        selection_descriptor,
        "pinned_v4_router_model_selection_seal.json",
        expected_artifact_types=SELECTION_ARTIFACT_TYPES,
        exact_object_name=parse_gs_uri(selection_descriptor["uri"])[1],
    )
    if sha256_file(selection_path) != prediction_seal["selection_seal_sha256"]:
        raise AssertionError("prediction/selection seal SHA mismatch")
    (
        winner,
        model_vocabulary,
        evaluation_vocabulary,
        _,
    ) = validate_selection(selection)
    router_fingerprint = router_architecture_fingerprint(winner)
    if winner["model_id"] != prediction_seal["winner_model_id"]:
        raise AssertionError("prediction/selection winner mismatch")
    if prediction_seal["winner_kind"] != "type_router":
        raise AssertionError("prediction winner kind mismatch")
    if router_fingerprint != prediction_seal["architecture_fingerprint"]:
        raise AssertionError("prediction/selection architecture mismatch")
    if winner["components"] != prediction_seal["components"]:
        raise AssertionError("prediction/selection component bundle mismatch")
    if winner["checkpoint_bundle_sha256"] != prediction_seal[
        "checkpoint_bundle_sha256"
    ]:
        raise AssertionError("prediction/selection bundle SHA mismatch")
    if canonical_sha256(prediction_seal["components"]) != prediction_seal[
        "checkpoint_bundle_sha256"
    ]:
        raise AssertionError("prediction component bundle hash mismatch")
    if winner["routing_contract"] != prediction_seal["routing_contract"]:
        raise AssertionError("prediction/selection routing contract mismatch")
    if winner["routing_sha256"] != prediction_seal["routing_sha256"]:
        raise AssertionError("prediction/selection routing SHA mismatch")
    if canonical_sha256(prediction_seal["routing_contract"]) != prediction_seal[
        "routing_sha256"
    ]:
        raise AssertionError("prediction routing hash mismatch")
    if dict(prediction_seal.get("route_by_type", {})) != FIXED_ROUTE_BY_TYPE:
        raise AssertionError("prediction fixed route mismatch")
    if prediction_seal.get("route_counts") != EXPECTED_ROUTE_COUNTS:
        raise AssertionError("prediction fixed route counts mismatch")
    if model_vocabulary != prediction_seal["model_obstacle_vocabulary"]:
        raise AssertionError("prediction/selection model vocabulary mismatch")
    if evaluation_vocabulary != prediction_seal["obstacle_vocabulary"]:
        raise AssertionError("prediction/selection evaluation vocabulary mismatch")

    inference_descriptor = prediction_seal["inference_image_seal"]
    inference_seal, inference_path = load_json_descriptor(
        bucket,
        inference_descriptor,
        "pinned_anonymous_test_image_seal.json",
        expected_artifact_types={"m3_inference_image_seal"},
        exact_object_name=parse_gs_uri(inference_descriptor["uri"])[1],
    )
    if sha256_file(inference_path) != prediction_seal["inference_image_seal_sha256"]:
        raise AssertionError("prediction/inference image seal SHA mismatch")
    if inference_seal.get("status") != "INFERENCE_IMAGES_COMPLETE":
        raise AssertionError("inference image seal status mismatch")
    if inference_seal.get("schema_version") != "dcc-m3-inference-image-seal-v1":
        raise AssertionError("inference image seal schema mismatch")
    if inference_seal.get("dataset_version") != EXPECTED_INFERENCE_DATASET_VERSION:
        raise AssertionError("inference image dataset version mismatch")
    if prediction_seal.get("anonymous_dataset_version") != (
        EXPECTED_INFERENCE_DATASET_VERSION
    ):
        raise AssertionError("prediction anonymous dataset version mismatch")
    if inference_seal.get("allowed_metadata") != ["sample_id"]:
        raise AssertionError("inference metadata allowlist mismatch")
    # The pinned v1 seal predates these two denylist names.  This is only
    # a declaration check; actual shard metadata still permits sample_id only
    # and is checked against the complete FORBIDDEN_META_KEYS set below.
    if not FORBIDDEN_META_KEYS.difference({"scene_group_id", "annotations"}).issubset(
        set(inference_seal.get("forbidden_metadata", []))
    ):
        raise AssertionError("inference metadata denylist is incomplete")
    require_all_assertions(inference_seal, "inference image seal")
    if inference_seal["test_sample_id_sha256"] != prediction_seal[
        "test_sample_id_sha256"
    ]:
        raise AssertionError("prediction/inference sample set mismatch")
    inference_seal_sha = sha256_file(inference_path)
    (
        dataset_seal,
        dataset_descriptor,
        dataset_path,
    ) = load_and_validate_selection_dataset(
        bucket,
        selection,
        inference_seal,
        inference_descriptor,
        inference_seal_sha,
    )
    if prediction_seal.get("dataset_seal") != dataset_descriptor:
        raise AssertionError("prediction/dataset descriptor mismatch")
    if prediction_seal.get("dataset_seal_sha256") != sha256_file(dataset_path):
        raise AssertionError("prediction/dataset seal SHA mismatch")
    if prediction_seal.get("dataset_version") != dataset_seal["dataset_version"]:
        raise AssertionError("prediction/dataset version mismatch")

    components = prediction_seal["components"]
    checkpoint_map = prediction_seal.get("component_checkpoints", {})
    expected_checkpoint_map = {
        "full_frame_regressor": components["full_frame_regressor"]["checkpoint"],
        "yolo_detector": components["yolo_detector"],
        "roi_regressor": components["roi_regressor"]["checkpoint"],
    }
    if checkpoint_map != expected_checkpoint_map:
        raise AssertionError("prediction component checkpoint map mismatch")
    verify_remote_descriptor(
        bucket,
        checkpoint_map["full_frame_regressor"],
        expected_artifact_types=REGRESSOR_CHECKPOINT_ARTIFACT_TYPES,
        expected_object_prefix="dcc/mission3/v4/runs/",
    )
    verify_remote_descriptor(
        bucket,
        checkpoint_map["roi_regressor"],
        expected_artifact_types=REGRESSOR_CHECKPOINT_ARTIFACT_TYPES,
        expected_object_prefix="dcc/mission3/v4/runs/",
    )
    verify_remote_descriptor(
        bucket,
        checkpoint_map["yolo_detector"],
        expected_artifact_types=DETECTOR_CHECKPOINT_ARTIFACT_TYPES,
        expected_object_prefix="dcc/mission3/v4/runs/",
    )

    inference_contract = prediction_seal.get("inference_contract", {})
    if canonical_sha256(inference_contract) != prediction_seal.get(
        "inference_contract_sha256"
    ):
        raise AssertionError("inference contract SHA mismatch")
    expected_inference_fields = {
        "schema_version": "dcc-m3-v4-router-test-inference-contract-v2",
        "selection_seal": selection_descriptor,
        "selection_seal_sha256": prediction_seal["selection_seal_sha256"],
        "dataset_seal": dataset_descriptor,
        "dataset_seal_sha256": prediction_seal["dataset_seal_sha256"],
        "dataset_version": prediction_seal["dataset_version"],
        "inference_image_seal": inference_descriptor,
        "inference_image_seal_sha256": prediction_seal[
            "inference_image_seal_sha256"
        ],
        "test_sample_id_sha256": prediction_seal["test_sample_id_sha256"],
        "winner_model_id": prediction_seal["winner_model_id"],
        "winner_kind": "type_router",
        "architecture_fingerprint": prediction_seal[
            "architecture_fingerprint"
        ],
        "components": components,
        "component_checkpoints": checkpoint_map,
        "checkpoint_bundle_sha256": prediction_seal[
            "checkpoint_bundle_sha256"
        ],
        "routing_contract": prediction_seal["routing_contract"],
        "routing_sha256": prediction_seal["routing_sha256"],
        "input_contract": ["images", "type_index"],
        "model_obstacle_vocabulary_sha256": prediction_seal[
            "model_obstacle_vocabulary_sha256"
        ],
        "evaluation_obstacle_vocabulary_sha256": prediction_seal[
            "obstacle_vocabulary_sha256"
        ],
    }
    mismatches = {
        key: (inference_contract.get(key), value)
        for key, value in expected_inference_fields.items()
        if inference_contract.get(key) != value
    }
    if mismatches:
        raise AssertionError(f"inference contract mismatch: {mismatches}")

    lease_descriptor = prediction_seal["test_use_lease"]
    lease, _ = load_json_descriptor(
        bucket,
        lease_descriptor,
        "pinned_v4_router_test_use_lease.json",
        expected_artifact_types={"m3_v4_router_test_use_lease"},
        exact_object_name=parse_gs_uri(lease_descriptor["uri"])[1],
    )
    expected_lease = {
        "status": "TEST_USE_RECORDED",
        "schema_version": "dcc-m3-v4-router-test-use-lease-v2",
        "test_use_policy": prediction_seal["test_use_policy"],
        "claim_scope": prediction_seal["claim_scope"],
        "prediction_run_id": prediction_seal["prediction_run_id"],
        "test_sample_id_sha256": prediction_seal["test_sample_id_sha256"],
        "selection_seal_sha256": prediction_seal["selection_seal_sha256"],
        "winner_model_id": prediction_seal["winner_model_id"],
        "checkpoint_bundle_sha256": prediction_seal[
            "checkpoint_bundle_sha256"
        ],
        "routing_sha256": prediction_seal["routing_sha256"],
        "inference_contract_sha256": prediction_seal[
            "inference_contract_sha256"
        ],
    }
    lease_mismatches = {
        key: (lease.get(key), expected)
        for key, expected in expected_lease.items()
        if lease.get(key) != expected
    }
    if lease_mismatches:
        raise AssertionError(f"test-use lease mismatch: {lease_mismatches}")

    attempt_bucket, attempt_object = parse_gs_uri(
        prediction_seal["attempt_prefix"]
    )
    if attempt_bucket != BUCKET_NAME:
        raise AssertionError("prediction attempt bucket mismatch")
    attempt_prefix = attempt_object.rstrip("/") + "/"
    prediction_path = download_descriptor(
        bucket,
        prediction_seal["prediction"],
        LOCAL_INPUT / "test_predictions.csv",
        expected_artifact_types={"m3_v4_router_anonymous_test_predictions"},
        expected_object_prefix=attempt_prefix,
    )
    diagnostics_path = download_descriptor(
        bucket,
        prediction_seal["diagnostics"],
        LOCAL_INPUT / "test_diagnostics.parquet",
        expected_artifact_types={"m3_v4_router_anonymous_test_diagnostics"},
        expected_object_prefix=attempt_prefix,
    )
    predictions = pd.read_csv(prediction_path).sort_values(
        "sample_id", kind="mergesort"
    ).reset_index(drop=True)
    diagnostics = pd.read_parquet(diagnostics_path).sort_values(
        "sample_id", kind="mergesort"
    ).reset_index(drop=True)
    expected_diagnostic_columns = [
        "sample_id",
        "route_component",
        "detector_conf",
        "n_candidates",
        "n_raw_candidates",
        "n_invalid_candidates",
        "fallback_used",
        "candidate_boxes_xyxy_json",
        "candidate_confidences_json",
        "obstacle_probabilities_json",
    ]
    if list(predictions.columns) != ["sample_id", "type", "pred_m"]:
        raise AssertionError("prediction schema changed")
    if prediction_seal.get("columns") != ["sample_id", "type", "pred_m"]:
        raise AssertionError("prediction seal column contract changed")
    if list(diagnostics.columns) != expected_diagnostic_columns:
        raise AssertionError("diagnostic schema changed")
    if list(diagnostics.columns) != prediction_seal["diagnostics_columns"]:
        raise AssertionError("diagnostic seal column contract changed")
    if len(predictions) != int(prediction_seal["row_count"]):
        raise AssertionError("prediction row count changed")
    if len(diagnostics) != len(predictions):
        raise AssertionError("diagnostic row count changed")
    if not predictions["sample_id"].is_unique or not diagnostics["sample_id"].is_unique:
        raise AssertionError("prediction or diagnostic ID duplicate")
    if predictions["sample_id"].tolist() != diagnostics["sample_id"].tolist():
        raise AssertionError("prediction/diagnostic ID mismatch")
    if predictions["type"].tolist() != predictions["sample_id"].str[0].tolist():
        raise AssertionError("prediction type provenance changed")
    if not np.isfinite(predictions["pred_m"].to_numpy(float)).all():
        raise AssertionError("prediction contains non-finite values")
    observed_type_counts = {
        str(key): int(value)
        for key, value in predictions["type"].value_counts().sort_index().items()
    }
    if observed_type_counts != prediction_seal.get("type_counts"):
        raise AssertionError("prediction type counts changed")

    expected_routes = predictions["type"].map(FIXED_ROUTE_BY_TYPE)
    if diagnostics["route_component"].tolist() != expected_routes.tolist():
        raise AssertionError("diagnostic route changed")
    if diagnostics["route_component"].value_counts().to_dict() != EXPECTED_ROUTE_COUNTS:
        raise AssertionError("diagnostic route counts changed")
    full_mask = diagnostics["route_component"].eq("full_frame")
    if diagnostics.loc[full_mask, "fallback_used"].astype(bool).any():
        raise AssertionError("full-frame route fallback flag changed")
    if not diagnostics.loc[full_mask, "candidate_boxes_xyxy_json"].eq("[]").all():
        raise AssertionError("full-frame candidate boxes changed")
    if not diagnostics.loc[full_mask, "candidate_confidences_json"].eq("[]").all():
        raise AssertionError("full-frame candidate confidences changed")
    if not diagnostics.loc[full_mask, "n_candidates"].eq(0).all():
        raise AssertionError("full-frame candidate count changed")
    if not (
        diagnostics["n_raw_candidates"]
        == diagnostics["n_candidates"] + diagnostics["n_invalid_candidates"]
    ).all():
        raise AssertionError("candidate count identity changed")
    for row in diagnostics.loc[~full_mask].itertuples(index=False):
        boxes = json.loads(row.candidate_boxes_xyxy_json)
        confidences = json.loads(row.candidate_confidences_json)
        if len(boxes) != int(row.n_candidates) or len(confidences) != int(
            row.n_candidates
        ):
            raise AssertionError("candidate diagnostic length mismatch")
        if any(len(box) != 4 for box in boxes):
            raise AssertionError("candidate bbox shape mismatch")
        if any(not math.isfinite(float(value)) for box in boxes for value in box):
            raise AssertionError("candidate bbox contains non-finite value")
        if any(
            not math.isfinite(float(value)) or not 0.0 <= float(value) <= 1.0
            for value in confidences
        ):
            raise AssertionError("candidate confidence invalid")

    parsed_probabilities = diagnostics["obstacle_probabilities_json"].map(json.loads)
    if not parsed_probabilities.map(len).eq(len(model_vocabulary)).all():
        raise AssertionError("diagnostic obstacle probability length changed")
    if model_vocabulary:
        probability_matrix = np.asarray(parsed_probabilities.tolist(), dtype=float)
        if not np.isfinite(probability_matrix).all() or not (
            (probability_matrix >= 0.0) & (probability_matrix <= 1.0)
        ).all():
            raise AssertionError("diagnostic obstacle probability is invalid")
    if canonical_sha256(sorted(predictions["sample_id"].tolist())) != prediction_seal[
        "test_sample_id_sha256"
    ]:
        raise AssertionError("prediction sample set changed")
    if sha256_file(prediction_path) != prediction_seal["prediction"]["sha256"]:
        raise AssertionError("prediction artifact changed")

    verification = {
        "prediction_seal_sha256": sha256_file(prediction_seal_path),
        "selection_seal_sha256": sha256_file(selection_path),
        "dataset_seal_sha256": sha256_file(dataset_path),
        "dataset_test_contract": dict(dataset_seal["test_contract"]),
        "dataset_test_contract_sha256": canonical_sha256(
            dataset_seal["test_contract"]
        ),
        "dataset_private_test_eval_contract": dict(
            dataset_seal["private"]["test_eval_contract"]
        ),
        "inference_image_seal_sha256": sha256_file(inference_path),
        "prediction_sha256": sha256_file(prediction_path),
        "test_sample_id_sha256": prediction_seal["test_sample_id_sha256"],
        "row_count": len(predictions),
        "routing_sha256": prediction_seal["routing_sha256"],
        "checkpoint_bundle_sha256": prediction_seal[
            "checkpoint_bundle_sha256"
        ],
    }
    return (
        prediction_seal,
        prediction_seal_descriptor,
        selection,
        inference_seal,
        predictions,
        diagnostics,
        verification,
        prediction_path,
    )


## Private metadata validation and metric functions.


In [ ]:
PRIVATE_REQUIRED_COLUMNS = [
    "sample_id",
    "stem",
    "type",
    "waterlevel_m",
    "obstruction",
    "none_only",
    "multi_obstacle",
    "contradiction",
    "empty_or_unknown",
    "group_obstruction_target_conflict",
    "group_id",
]


def normalized_obstruction(value: Any, vocabulary: Sequence[str]) -> list[str]:
    if isinstance(value, np.ndarray):
        values = value.tolist()
    elif isinstance(value, (list, tuple)):
        values = list(value)
    elif value is None or (isinstance(value, float) and math.isnan(value)):
        values = []
    else:
        raise AssertionError(f"obstruction must be a list-like value: {type(value)}")
    labels = [str(item) for item in values]
    if len(labels) != len(set(labels)):
        raise AssertionError("duplicate obstruction label in private metadata")
    vocabulary_index = {label: index for index, label in enumerate(vocabulary)}
    unknown = set(labels) - set(vocabulary_index)
    if unknown:
        raise AssertionError(f"unknown obstruction labels: {sorted(unknown)}")
    if labels != sorted(labels, key=vocabulary_index.__getitem__):
        raise AssertionError("obstruction labels are not in frozen vocabulary order")
    return labels


def validate_private_metadata(
    frame: pd.DataFrame,
    contract: Mapping[str, Any],
    vocabulary: Sequence[str],
) -> pd.DataFrame:
    contract_columns = [str(value) for value in contract.get("columns", [])]
    if contract_columns != PRIVATE_REQUIRED_COLUMNS:
        raise AssertionError("private contract columns changed")
    if frame.columns.tolist() != contract_columns:
        raise AssertionError(
            "private metadata columns/order differ from the sealed contract"
        )
    frame = frame.copy()
    if len(frame) != int(contract["row_count"]):
        raise AssertionError("private metadata row count mismatch")
    if not frame["sample_id"].is_unique or not frame["stem"].is_unique:
        raise AssertionError("private sample_id/stem must be unique")
    if frame["sample_id"].isna().any() or frame["stem"].isna().any():
        raise AssertionError("private key contains null")
    if not frame["sample_id"].astype(str).map(
        lambda value: bool(SAFE_ID_PATTERN.fullmatch(value))
    ).all():
        raise AssertionError("private sample_id format mismatch")
    frame["sample_id"] = frame["sample_id"].astype(str)
    frame["stem"] = frame["stem"].astype(str)
    frame["type"] = frame["type"].astype(str)
    if not frame["type"].isin(TYPE_ORDER).all():
        raise AssertionError("private type value mismatch")
    if not frame["type"].eq(frame["sample_id"].str[0]).all():
        raise AssertionError("private type/sample_id mismatch")
    if not frame["type"].eq(frame["stem"].str[0]).all():
        raise AssertionError("private type/stem mismatch")
    try:
        frame["waterlevel_m"] = pd.to_numeric(
            frame["waterlevel_m"], errors="raise"
        ).astype(float)
    except (TypeError, ValueError) as error:
        raise AssertionError("private target must be numeric") from error
    if not np.isfinite(frame["waterlevel_m"].to_numpy(float)).all():
        raise AssertionError("private target contains non-finite values")
    for type_name, (low, high) in DEFAULT_TYPE_RANGES.items():
        values = frame.loc[frame["type"].eq(type_name), "waterlevel_m"]
        if values.empty or not values.between(low, high).all():
            raise AssertionError(f"private target range mismatch: {type_name}")
    if frame["group_id"].isna().any() or frame["group_id"].astype(str).eq("").any():
        raise AssertionError("private group_id missing")
    frame["group_id"] = frame["group_id"].astype(str)
    for flag in [
        "none_only",
        "multi_obstacle",
        "contradiction",
        "empty_or_unknown",
        "group_obstruction_target_conflict",
    ]:
        if frame[flag].isna().any():
            raise AssertionError(f"private flag contains null: {flag}")
        if not pd.api.types.is_bool_dtype(frame[flag].dtype):
            raise AssertionError(f"private flag must have boolean dtype: {flag}")
    frame["obstruction"] = frame["obstruction"].map(
        lambda value: normalized_obstruction(value, vocabulary)
    )
    lengths = frame["obstruction"].map(len)
    if not frame["multi_obstacle"].eq(lengths.ge(2)).all():
        raise AssertionError("multi_obstacle flag mismatch")
    # 04A removes raw none/empty/unknown tokens from this field and preserves
    # their independent provenance in the sealed boolean flags.
    if (frame["none_only"] & lengths.gt(0)).any():
        raise AssertionError("none_only row contains a real obstruction")
    if (frame["contradiction"] & lengths.eq(0)).any():
        raise AssertionError("contradiction row lacks a real obstruction")
    if (frame["none_only"] & frame["contradiction"]).any():
        raise AssertionError("none_only and contradiction cannot both be true")
    if (lengths.eq(0) & ~(frame["none_only"] | frame["empty_or_unknown"])).any():
        raise AssertionError("zero-label row lacks none/unknown provenance")
    observed_sha = canonical_sha256(sorted(frame["sample_id"].tolist()))
    if observed_sha != contract["test_sample_id_sha256"]:
        raise AssertionError("private metadata sample set mismatch")
    return frame


def group_bootstrap_rmse_ci(
    part: pd.DataFrame,
    group_column: str,
    *,
    reps: int,
    seed: int,
) -> tuple[float | None, float | None]:
    if reps <= 0:
        return None, None
    grouped = (
        part.assign(error_sq=part["error_m"].to_numpy(float) ** 2)
        .groupby(group_column, observed=True)["error_sq"]
        .agg(["sum", "count"])
    )
    if len(grouped) < 2:
        return None, None
    sums = grouped["sum"].to_numpy(float)
    counts = grouped["count"].to_numpy(float)
    rng = np.random.default_rng(seed)
    values = np.empty(reps, dtype=float)
    group_count = len(grouped)
    for index in range(reps):
        sampled = rng.integers(0, group_count, size=group_count)
        values[index] = math.sqrt(float(sums[sampled].sum() / counts[sampled].sum()))
    low, high = np.quantile(values, [0.025, 0.975])
    return float(low), float(high)


def summarize_metric(
    part: pd.DataFrame,
    *,
    scope_kind: str,
    scope_value: str,
    group_column: str,
) -> dict[str, Any]:
    error = part["error_m"].to_numpy(float)
    scope_seed = BOOTSTRAP_SEED + int(
        hashlib.sha256(f"{scope_kind}:{scope_value}".encode("utf-8")).hexdigest()[:8],
        16,
    )
    ci_low, ci_high = group_bootstrap_rmse_ci(
        part,
        group_column,
        reps=BOOTSTRAP_REPS,
        seed=scope_seed,
    )
    return {
        "scope_kind": scope_kind,
        "scope_value": scope_value,
        "n": int(len(part)),
        "n_groups": int(part[group_column].nunique()),
        "rmse_m": float(np.sqrt(np.mean(error**2))),
        "rmse_ci95_low_m": ci_low,
        "rmse_ci95_high_m": ci_high,
        "mae_m": float(np.mean(np.abs(error))),
        "bias_m": float(np.mean(error)),
        "p95_abs_error_m": float(np.quantile(np.abs(error), 0.95)),
    }


def build_metrics(
    joined: pd.DataFrame,
    vocabulary: Sequence[str],
    group_column: str,
) -> pd.DataFrame:
    rows = [
        summarize_metric(
            joined,
            scope_kind="overall",
            scope_value="ALL",
            group_column=group_column,
        )
    ]
    for type_name in TYPE_ORDER:
        part = joined.loc[joined["type"].eq(type_name)]
        if part.empty:
            raise AssertionError(f"test type missing: {type_name}")
        rows.append(
            summarize_metric(
                part,
                scope_kind="type",
                scope_value=type_name,
                group_column=group_column,
            )
        )
    obstacle_target_known = ~joined["group_obstruction_target_conflict"].astype(
        bool
    )
    for obstacle in vocabulary:
        obstacle_mask = (
            obstacle_target_known
            & joined["obstruction"].map(lambda labels: obstacle in labels)
        )
        part = joined.loc[obstacle_mask]
        if len(part) >= MIN_OBSTACLE_SLICE_N:
            rows.append(
                summarize_metric(
                    part,
                    scope_kind="obstacle",
                    scope_value=obstacle,
                    group_column=group_column,
                )
            )
        for type_name in TYPE_ORDER:
            joint = joined.loc[
                obstacle_mask & joined["type"].eq(type_name)
            ]
            if len(joint) < MIN_OBSTACLE_SLICE_N:
                continue
            rows.append(
                summarize_metric(
                    joint,
                    scope_kind="type_obstacle",
                    scope_value=f"{type_name}|{obstacle}",
                    group_column=group_column,
                )
            )
    return pd.DataFrame(rows)


def auxiliary_obstacle_metrics(
    joined: pd.DataFrame,
    vocabulary: Sequence[str],
    thresholds: Sequence[float],
) -> pd.DataFrame:
    if not vocabulary:
        return pd.DataFrame(
            columns=[
                "obstacle",
                "n_known",
                "n_positive",
                "threshold",
                "average_precision",
                "f1",
            ]
        )
    if len(thresholds) != len(vocabulary):
        raise AssertionError("frozen obstacle threshold count mismatch")
    parsed = joined["obstacle_probabilities_json"].map(json.loads)
    if not parsed.map(len).eq(len(vocabulary)).all():
        raise AssertionError("diagnostic obstacle probability length mismatch")
    probabilities = np.asarray(parsed.tolist(), dtype=float)
    if not np.isfinite(probabilities).all() or not (
        (probabilities >= 0.0) & (probabilities <= 1.0)
    ).all():
        raise AssertionError("diagnostic obstacle probabilities invalid")
    target = np.asarray(
        [
            [float(obstacle in labels) for obstacle in vocabulary]
            for labels in joined["obstruction"]
        ],
        dtype=float,
    )
    # The training mask excludes rows whose raw tag state is unknown,
    # contradictory, or inconsistent within an image group. Apply the same
    # contract to auxiliary test classification.
    known_rows = ~(
        joined["empty_or_unknown"].to_numpy(bool)
        | joined["contradiction"].to_numpy(bool)
        | joined["group_obstruction_target_conflict"].to_numpy(bool)
    )
    if not known_rows.any():
        raise AssertionError("no known rows for auxiliary obstacle evaluation")
    rows = []
    for index, obstacle in enumerate(vocabulary):
        y_true = target[known_rows, index].astype(int)
        y_score = probabilities[known_rows, index]
        ap = (
            float(average_precision_score(y_true, y_score))
            if 0 < y_true.sum() < len(y_true)
            else None
        )
        y_pred = (y_score >= float(thresholds[index])).astype(int)
        rows.append(
            {
                "obstacle": obstacle,
                "n_known": int(len(y_true)),
                "n_positive": int(y_true.sum()),
                "threshold": float(thresholds[index]),
                "average_precision": ap,
                "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            }
        )
    return pd.DataFrame(rows)


def json_safe_number(value: Any) -> float | int | None:
    if value is None:
        return None
    if isinstance(value, (np.integer, int)):
        return int(value)
    numeric = float(value)
    return numeric if math.isfinite(numeric) else None


def metric_payload(metrics: pd.DataFrame) -> dict[str, Any]:
    payload: dict[str, Any] = {}
    for row in metrics.to_dict(orient="records"):
        key = f"{row['scope_kind']}:{row['scope_value']}"
        payload[key] = {
            column: json_safe_number(row[column])
            for column in [
                "n",
                "n_groups",
                "rmse_m",
                "rmse_ci95_low_m",
                "rmse_ci95_high_m",
                "mae_m",
                "bias_m",
                "p95_abs_error_m",
            ]
        }
    return payload


def save_metric_figures(metrics: pd.DataFrame, joined: pd.DataFrame) -> dict[str, Path]:
    type_rows = metrics.loc[metrics["scope_kind"].eq("type")].copy()
    type_rows["scope_value"] = pd.Categorical(
        type_rows["scope_value"], categories=TYPE_ORDER, ordered=True
    )
    type_rows = type_rows.sort_values("scope_value")
    overall = float(
        metrics.loc[metrics["scope_kind"].eq("overall"), "rmse_m"].iloc[0]
    )
    type_path = LOCAL_FIGURES / "test_rmse_by_type.png"
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(type_rows["scope_value"].astype(str), type_rows["rmse_m"], color="#4C78A8")
    ax.axhline(overall, color="#E45756", linestyle="--", label=f"ALL {overall:.3f} m")
    ax.set_xlabel("Mokja type")
    ax.set_ylabel("RMSE (m)")
    ax.set_title("Mission 3 v4 test RMSE by type")
    ax.legend()
    fig.tight_layout()
    fig.savefig(type_path, dpi=160)
    plt.close(fig)

    obstacle_rows = metrics.loc[metrics["scope_kind"].eq("obstacle")].sort_values(
        "rmse_m", ascending=True
    )
    obstacle_path = LOCAL_FIGURES / "test_rmse_by_obstacle.png"
    fig_height = max(5.0, 0.42 * max(1, len(obstacle_rows)))
    fig, ax = plt.subplots(figsize=(11, fig_height))
    ax.barh(obstacle_rows["scope_value"], obstacle_rows["rmse_m"], color="#72B7B2")
    ax.axvline(overall, color="#E45756", linestyle="--", label=f"ALL {overall:.3f} m")
    for position, row in enumerate(obstacle_rows.itertuples(index=False)):
        ax.text(float(row.rmse_m), position, f"  n={int(row.n)}", va="center", fontsize=8)
    ax.set_xlabel("RMSE (m); obstacle slices overlap")
    ax.set_ylabel("Obstacle")
    ax.set_title("Mission 3 v4 test RMSE by obstacle")
    ax.legend()
    fig.tight_layout()
    fig.savefig(obstacle_path, dpi=160)
    plt.close(fig)

    residual_path = LOCAL_FIGURES / "test_residual_distribution.png"
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(joined["error_m"], bins=80, color="#54A24B", alpha=0.9)
    ax.axvline(0.0, color="black", linewidth=1)
    ax.set_xlabel("Prediction - truth (m)")
    ax.set_ylabel("Images")
    ax.set_title("Mission 3 v4 test residual distribution")
    fig.tight_layout()
    fig.savefig(residual_path, dpi=160)
    plt.close(fig)
    return {
        "type_rmse_figure": type_path,
        "obstacle_rmse_figure": obstacle_path,
        "residual_figure": residual_path,
    }


## Preserved 03F router/fallback/error-slice analysis.


In [ ]:
def build_router_diagnostic_slices(
    joined: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required = {
        "sample_id",
        "stem",
        "type",
        "waterlevel_m",
        "pred_m",
        "error_m",
        "abs_error_m",
        "obstruction",
        "contradiction",
        "route_component",
        "fallback_used",
        "n_candidates",
        "detector_conf",
    }
    missing = required - set(joined.columns)
    if missing:
        raise AssertionError(f"03F diagnostic columns missing: {sorted(missing)}")
    enriched = joined.copy()
    type_quantiles = (
        enriched.groupby("type", observed=True)["waterlevel_m"]
        .quantile([0.1, 0.9])
        .unstack()
    )
    occlusion_tags = {"수풀", "부착물", "부유물", "사람", "반투명"}
    water_tags = {"거센물살", "와류", "탁류", "거울반사", "달빛반사", "야간", "안개"}

    def hypothesis_tags(row: pd.Series) -> str:
        hypotheses: list[str] = []
        tags = set(str(value) for value in row["obstruction"])
        if bool(row["fallback_used"]):
            hypotheses.extend(["detector_miss", "fallback_used"])
        if bool(row["contradiction"]):
            hypotheses.append("label_uncertainty")
        low = float(type_quantiles.loc[row["type"], 0.1])
        high = float(type_quantiles.loc[row["type"], 0.9])
        if (
            row["type"] in {"F", "G", "H"}
            or float(row["waterlevel_m"]) <= low
            or float(row["waterlevel_m"]) >= high
        ):
            hypotheses.append("rare_type_or_extreme_level")
        if tags.intersection(occlusion_tags):
            hypotheses.append("occlusion")
        if tags.intersection(water_tags):
            hypotheses.append("waterline_ambiguity")
        if not hypotheses:
            hypotheses.append("requires_visual_review")
        return ";".join(dict.fromkeys(hypotheses))

    enriched["hypothesis_tags"] = enriched.apply(hypothesis_tags, axis=1)

    def summarize_slice(
        frame: pd.DataFrame,
        dimension: str,
        value: Any,
    ) -> dict[str, Any]:
        error = frame["error_m"].to_numpy(float)
        if not len(error):
            raise AssertionError("empty diagnostic slice")
        return {
            "dimension": dimension,
            "value": str(value),
            "n": int(len(frame)),
            "rmse_m": float(np.sqrt(np.mean(error**2))),
            "mae_m": float(np.mean(np.abs(error))),
            "bias_m": float(np.mean(error)),
        }

    slice_rows: list[dict[str, Any]] = []
    for value, part in enriched.groupby("route_component", dropna=False):
        slice_rows.append(summarize_slice(part, "route_component", value))
    for value, part in enriched.groupby("fallback_used", dropna=False):
        slice_rows.append(summarize_slice(part, "fallback_used", value))

    enriched["candidate_count_slice"] = pd.cut(
        enriched["n_candidates"].fillna(0).astype(int),
        bins=[-1, 0, 1, 3, float("inf")],
        labels=["0", "1", "2-3", "4+"],
    ).astype(str)
    for value, part in enriched.groupby("candidate_count_slice", dropna=False):
        slice_rows.append(summarize_slice(part, "candidate_count", value))

    enriched["detector_confidence_slice"] = pd.cut(
        enriched["detector_conf"],
        bins=[-float("inf"), 0.10, 0.25, 0.50, float("inf")],
        labels=["<=0.10", "0.10-0.25", "0.25-0.50", ">0.50"],
    ).astype(str)
    enriched.loc[
        enriched["detector_conf"].isna(), "detector_confidence_slice"
    ] = "not_applicable"
    for value, part in enriched.groupby(
        "detector_confidence_slice", dropna=False
    ):
        slice_rows.append(
            summarize_slice(part, "detector_confidence", value)
        )

    enriched["level_decile_within_type"] = (
        enriched.groupby("type", observed=True)["waterlevel_m"]
        .transform(
            lambda values: pd.qcut(
                values.rank(method="first"), 10, labels=False
            )
        )
        .astype(int)
    )
    for (type_name, decile), part in enriched.groupby(
        ["type", "level_decile_within_type"], observed=True
    ):
        slice_rows.append(
            summarize_slice(
                part,
                "type_level_decile",
                f"{type_name}:D{int(decile)}",
            )
        )

    enriched["has_obstruction"] = enriched["obstruction"].map(bool)
    for value, part in enriched.groupby("has_obstruction", dropna=False):
        slice_rows.append(summarize_slice(part, "has_obstruction", value))
    obstruction_long = enriched[
        ["sample_id", "error_m", "obstruction"]
    ].explode("obstruction")
    obstruction_long = obstruction_long.loc[
        obstruction_long["obstruction"].notna()
    ]
    for value, part in obstruction_long.groupby("obstruction", dropna=False):
        if len(part) >= MIN_OBSTACLE_SLICE_N:
            slice_rows.append(summarize_slice(part, "obstruction", value))

    performance_slices = pd.DataFrame(slice_rows).sort_values(
        ["dimension", "rmse_m"],
        ascending=[True, False],
        kind="mergesort",
    ).reset_index(drop=True)

    selection_reasons: defaultdict[str, set[str]] = defaultdict(set)

    def add_reason(sample_ids: Iterable[Any], reason: str) -> None:
        for sample_id in sample_ids:
            selection_reasons[str(sample_id)].add(reason)

    add_reason(enriched.nlargest(20, "abs_error_m")["sample_id"], "global_top20")
    for type_index, type_name in enumerate(TYPE_ORDER):
        part = enriched.loc[enriched["type"].eq(type_name)]
        add_reason(
            part.nlargest(5, "abs_error_m")["sample_id"],
            f"{type_name}_top5",
        )
        threshold = float(part["abs_error_m"].quantile(0.90))
        high = part.loc[part["abs_error_m"].ge(threshold)]
        sampled = high.sample(
            n=min(3, len(high)),
            random_state=42 + type_index,
            replace=False,
        )
        add_reason(
            sampled["sample_id"],
            f"{type_name}_top10pct_seed42",
        )
    fallback_rows = enriched.loc[enriched["fallback_used"].astype(bool)]
    add_reason(
        fallback_rows.nlargest(
            min(10, len(fallback_rows)), "abs_error_m"
        )["sample_id"],
        "fallback_top_error",
    )
    error_selection = enriched.loc[
        enriched["sample_id"].isin(selection_reasons)
    ].copy()
    error_selection["selection_reason"] = error_selection["sample_id"].map(
        lambda value: ";".join(sorted(selection_reasons[str(value)]))
    )
    error_selection = error_selection.sort_values(
        ["abs_error_m", "sample_id"],
        ascending=[False, True],
        kind="mergesort",
    ).reset_index(drop=True)
    return enriched, performance_slices, error_selection


## EVALUATION phase: freeze plan, then access truth and publish final seal.


In [ ]:
def run_evaluation() -> None:
    _, bucket = authenticate_storage()
    (
        prediction_seal,
        prediction_seal_descriptor,
        selection,
        inference_seal,
        predictions,
        diagnostics,
        verification,
        prediction_path,
    ) = verify_prediction_chain_before_truth(bucket)
    (
        private_contract,
        private_contract_descriptor,
        private_contract_path,
    ) = load_private_eval_contract(bucket)
    if sha256_file(private_contract_path) != private_contract_descriptor["sha256"]:
        raise AssertionError("private eval-contract artifact SHA mismatch")
    dataset_private_descriptor = verification[
        "dataset_private_test_eval_contract"
    ]
    descriptor_keys = sorted(DESCRIPTOR_KEYS)
    if {
        key: dataset_private_descriptor[key] for key in descriptor_keys
    } != {
        key: private_contract_descriptor[key] for key in descriptor_keys
    }:
        raise AssertionError("dataset/private eval-contract descriptor mismatch")
    private_descriptor = private_contract["test_eval_metadata"]
    verify_remote_descriptor(
        bucket,
        private_descriptor,
        expected_artifact_types={"m3_v4_private_test_eval_metadata"},
        expected_object_prefix=PRIVATE_SPLIT_PREFIX,
    )
    if int(private_contract["row_count"]) != int(prediction_seal["row_count"]):
        raise AssertionError("private contract/prediction row count mismatch")
    if private_contract["test_sample_id_sha256"] != prediction_seal[
        "test_sample_id_sha256"
    ]:
        raise AssertionError("private contract/prediction sample set mismatch")
    if private_contract["obstacle_vocabulary"] != prediction_seal[
        "obstacle_vocabulary"
    ]:
        raise AssertionError("private contract/prediction obstacle vocabulary mismatch")
    selection_test_contract = selection["test_contract"]
    if dict(selection_test_contract) != verification["dataset_test_contract"]:
        raise AssertionError("selection/dataset test contract changed")
    if int(selection_test_contract["row_count"]) != int(
        private_contract["row_count"]
    ):
        raise AssertionError("dataset/private test row count mismatch")
    if selection_test_contract["test_sample_id_sha256"] != private_contract[
        "test_sample_id_sha256"
    ]:
        raise AssertionError("dataset/private sample-set SHA mismatch")
    if selection_test_contract["inference_image_seal_sha256"] != verification[
        "inference_image_seal_sha256"
    ]:
        raise AssertionError("dataset/inference seal SHA mismatch")

    evaluation_spec = {
        "schema_version": "dcc-m3-v4-evaluation-spec-v2",
        "prediction_columns": ["sample_id", "type", "pred_m"],
        "join": "one_to_one_outer_on_sample_id_all_rows_both",
        "target_column": "waterlevel_m",
        "error_definition": "pred_m-waterlevel_m",
        "overall_metric": "sqrt(mean(error_m**2)) over all rows",
        "partition_metrics": ["type"],
        "overlapping_slice_metrics": ["obstacle", "type_obstacle"],
        "obstacle_target_conflict_policy": {
            "flag_column": "group_obstruction_target_conflict",
            "overall_and_type_metrics": "include all rows",
            "obstacle_and_type_obstacle_metrics": "exclude flagged rows",
            "auxiliary_obstacle_metrics": "exclude flagged rows",
        },
        "router_diagnostic_slices": [
            "route_component",
            "fallback_used",
            "candidate_count",
            "detector_confidence",
            "type_level_decile",
            "has_obstruction",
            "obstruction",
        ],
        "error_selection_rules": [
            "global_top20",
            "type_top5",
            "type_top10pct_seed42_n3",
            "fallback_top10",
        ],
        "reported_metrics": [
            "n",
            "n_groups",
            "rmse_m",
            "rmse_ci95_low_m",
            "rmse_ci95_high_m",
            "mae_m",
            "bias_m",
            "p95_abs_error_m",
        ],
        "confidence_interval": {
            "method": "group bootstrap percentile",
            "group_column": "group_id",
            "repetitions": BOOTSTRAP_REPS,
            "seed": BOOTSTRAP_SEED,
            "coverage": 0.95,
        },
        "minimum_obstacle_slice_n": MIN_OBSTACLE_SLICE_N,
        "minimum_type_obstacle_slice_n": MIN_OBSTACLE_SLICE_N,
        "obstacle_slices_overlap": True,
        "type_obstacle_slices_overlap": True,
        "weighted_identity_applies_to": "type partition only",
        "auxiliary_obstacle_threshold_source": "frozen validation selection seal",
    }
    evaluation_plan = {
        "status": "EVALUATION_PLAN_FROZEN_BEFORE_TRUTH",
        "schema_version": "dcc-m3-v4-evaluation-plan-seal-v1",
        "evaluation_id": EVALUATION_ID,
        "claim_scope": CLAIM_SCOPE,
        "prediction_run_id": prediction_seal["prediction_run_id"],
        "prediction_seal": prediction_seal_descriptor,
        "prediction_seal_sha256": verification["prediction_seal_sha256"],
        "prediction_sha256": verification["prediction_sha256"],
        "selection_seal_sha256": verification["selection_seal_sha256"],
        "inference_image_seal_sha256": verification[
            "inference_image_seal_sha256"
        ],
        "test_sample_id_sha256": verification["test_sample_id_sha256"],
        "row_count": verification["row_count"],
        "private_eval_contract": private_contract_descriptor,
        "private_contract_sha256": private_contract_descriptor["sha256"],
        "private_contract_payload_sha256": canonical_sha256(private_contract),
        "private_contract_columns": private_contract["columns"],
        "private_test_eval_metadata": private_descriptor,
        "obstacle_vocabulary": prediction_seal["obstacle_vocabulary"],
        "obstacle_vocabulary_sha256": prediction_seal[
            "obstacle_vocabulary_sha256"
        ],
        "evaluation_spec": evaluation_spec,
        "evaluation_spec_sha256": canonical_sha256(evaluation_spec),
        "routing_sha256": prediction_seal["routing_sha256"],
        "checkpoint_bundle_sha256": prediction_seal[
            "checkpoint_bundle_sha256"
        ],
        "private_truth_content_accessed": False,
        "assertions": {
            "prediction_chain_verified": True,
            "selection_reverified": True,
            "dataset_test_contract_reverified": True,
            "dataset_private_contract_reverified": True,
            "inference_contract_reverified": True,
            "test_use_lease_reverified": True,
            "prediction_hash_verified": sha256_file(prediction_path)
            == prediction_seal["prediction"]["sha256"],
            "private_contract_matches_prediction": private_contract[
                "test_sample_id_sha256"
            ]
            == prediction_seal["test_sample_id_sha256"],
            "metric_rules_frozen": True,
            "truth_not_downloaded_yet": True,
        },
    }
    plan_descriptor = upload_json_create_only(
        bucket,
        evaluation_plan,
        f"{EVALUATION_PREFIX}/evaluation_plan_seal.json",
        "m3_v4_evaluation_plan_seal",
    )
    print("EVALUATION PLAN FROZEN — private truth download is now allowed")
    print("plan seal:", plan_descriptor["uri"])

    # First private-content read occurs only after the create-only plan succeeded.
    private_path = download_descriptor(
        bucket,
        private_descriptor,
        LOCAL_INPUT / "test_eval_metadata.parquet",
        expected_artifact_types={"m3_v4_private_test_eval_metadata"},
        expected_object_prefix=PRIVATE_SPLIT_PREFIX,
    )
    evaluation_vocabulary = [
        str(value) for value in prediction_seal["obstacle_vocabulary"]
    ]
    model_vocabulary = [
        str(value) for value in prediction_seal["model_obstacle_vocabulary"]
    ]
    private_metadata = validate_private_metadata(
        pd.read_parquet(private_path),
        private_contract,
        evaluation_vocabulary,
    )
    if sha256_file(prediction_path) != prediction_seal["prediction"]["sha256"]:
        raise AssertionError("prediction changed after private-data access")

    joined = predictions.merge(
        private_metadata,
        on="sample_id",
        how="outer",
        validate="one_to_one",
        indicator="_private_join",
        suffixes=("_prediction", "_private"),
    )
    if not joined["_private_join"].eq("both").all():
        counts = joined["_private_join"].value_counts().to_dict()
        raise AssertionError(f"private join incomplete: {counts}")
    if not joined["type_prediction"].eq(joined["type_private"]).all():
        raise AssertionError("prediction/private type mismatch")
    joined["type"] = joined["type_prediction"]
    joined = joined.drop(
        columns=["_private_join", "type_prediction", "type_private"]
    ).merge(
        diagnostics,
        on="sample_id",
        how="outer",
        validate="one_to_one",
        indicator="_diagnostic_join",
    )
    if not joined["_diagnostic_join"].eq("both").all():
        raise AssertionError("diagnostic join incomplete")
    joined = joined.drop(columns="_diagnostic_join")
    if len(joined) != int(prediction_seal["row_count"]):
        raise AssertionError("joined test row count mismatch")
    if not np.isfinite(joined[["pred_m", "waterlevel_m"]].to_numpy(float)).all():
        raise AssertionError("joined prediction/target is non-finite")
    joined["error_m"] = joined["pred_m"] - joined["waterlevel_m"]
    joined["abs_error_m"] = joined["error_m"].abs()
    joined, performance_slices, error_selection = (
        build_router_diagnostic_slices(joined)
    )

    group_column = "group_id"
    metrics = build_metrics(joined, evaluation_vocabulary, group_column)
    overall_rmse = float(
        metrics.loc[metrics["scope_kind"].eq("overall"), "rmse_m"].iloc[0]
    )
    type_metrics = metrics.loc[metrics["scope_kind"].eq("type")]
    type_weighted_mse = float(
        (type_metrics["n"] * type_metrics["rmse_m"] ** 2).sum()
        / type_metrics["n"].sum()
    )
    if not np.isclose(
        overall_rmse**2, type_weighted_mse, rtol=1e-12, atol=1e-14
    ):
        raise AssertionError("overall/type weighted MSE identity failed")

    winner = selection["winner"]
    if model_vocabulary:
        thresholds = winner.get("obstacle_thresholds")
        if thresholds is None:
            raise AssertionError(
                "multitask router must freeze validation obstacle thresholds"
            )
        aux_metrics = auxiliary_obstacle_metrics(
            joined, model_vocabulary, thresholds
        )
    else:
        aux_metrics = auxiliary_obstacle_metrics(joined, [], [])

    metrics_path = LOCAL_OUTPUT / "test_metrics.csv"
    residuals_path = LOCAL_OUTPUT / "test_residuals.parquet"
    aux_metrics_path = LOCAL_OUTPUT / "test_obstacle_aux_metrics.csv"
    slices_path = LOCAL_OUTPUT / "test_router_diagnostic_slices.csv"
    error_selection_path = LOCAL_OUTPUT / "test_error_selection.parquet"
    metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
    joined.to_parquet(residuals_path, index=False, compression="zstd")
    aux_metrics.to_csv(aux_metrics_path, index=False, encoding="utf-8-sig")
    performance_slices.to_csv(
        slices_path, index=False, encoding="utf-8-sig"
    )
    error_selection.to_parquet(
        error_selection_path, index=False, compression="zstd"
    )
    figure_paths = save_metric_figures(metrics, joined)

    artifact_specs: dict[str, tuple[Path, str, str]] = {
        "metrics": (
            metrics_path,
            "test_metrics.csv",
            "m3_v4_private_test_metrics",
        ),
        "residuals": (
            residuals_path,
            "test_residuals.parquet",
            "m3_v4_private_test_residuals",
        ),
        "obstacle_aux_metrics": (
            aux_metrics_path,
            "test_obstacle_aux_metrics.csv",
            "m3_v4_private_obstacle_aux_metrics",
        ),
        "router_diagnostic_slices": (
            slices_path,
            "test_router_diagnostic_slices.csv",
            "m3_v4_private_router_diagnostic_slices",
        ),
        "error_selection": (
            error_selection_path,
            "test_error_selection.parquet",
            "m3_v4_private_error_selection",
        ),
        "type_rmse_figure": (
            figure_paths["type_rmse_figure"],
            "figures/test_rmse_by_type.png",
            "m3_v4_private_figure",
        ),
        "obstacle_rmse_figure": (
            figure_paths["obstacle_rmse_figure"],
            "figures/test_rmse_by_obstacle.png",
            "m3_v4_private_figure",
        ),
        "residual_figure": (
            figure_paths["residual_figure"],
            "figures/test_residual_distribution.png",
            "m3_v4_private_figure",
        ),
    }
    artifact_descriptors = {
        name: upload_create_only(
            bucket,
            local_path,
            f"{EVALUATION_PREFIX}/{relative_name}",
            artifact_type,
        )
        for name, (local_path, relative_name, artifact_type) in artifact_specs.items()
    }
    if sha256_file(prediction_path) != prediction_seal["prediction"]["sha256"]:
        raise AssertionError("prediction changed during evaluation")

    evaluation_assertions = {
        "evaluation_plan_preceded_truth": True,
        "prediction_chain_verified_before_truth": True,
        "selection_reverified_before_truth": True,
        "join_one_to_one_complete": len(joined) == int(prediction_seal["row_count"]),
        "sample_id_set_exact": canonical_sha256(
            sorted(joined["sample_id"].astype(str).tolist())
        )
        == prediction_seal["test_sample_id_sha256"],
        "type_provenance_exact": joined["type"].tolist()
        == joined["sample_id"].str[0].tolist(),
        "all_values_finite": bool(
            np.isfinite(joined[["pred_m", "waterlevel_m", "error_m"]]).all().all()
        ),
        "overall_type_weighted_identity": bool(
            np.isclose(
                overall_rmse**2,
                type_weighted_mse,
                rtol=1e-12,
                atol=1e-14,
            )
        ),
        "obstacle_slices_marked_overlapping": evaluation_spec[
            "obstacle_slices_overlap"
        ],
        "type_obstacle_slices_marked_overlapping": evaluation_spec[
            "type_obstacle_slices_overlap"
        ],
        "type_obstacle_slices_generated": bool(
            metrics["scope_kind"].eq("type_obstacle").any()
        ),
        "route_exact_after_truth": joined["route_component"].tolist()
        == joined["type"].map(FIXED_ROUTE_BY_TYPE).tolist(),
        "route_counts_exact_after_truth": joined[
            "route_component"
        ].value_counts().to_dict()
        == EXPECTED_ROUTE_COUNTS,
        "full_route_fallback_false_after_truth": bool(
            ~joined.loc[
                joined["route_component"].eq("full_frame"),
                "fallback_used",
            ].astype(bool).any()
        ),
        "router_diagnostic_slices_generated": set(
            performance_slices["dimension"]
        )
        >= {
            "route_component",
            "fallback_used",
            "candidate_count",
            "detector_confidence",
            "type_level_decile",
        },
        "error_selection_rule_fixed": bool(len(error_selection) > 0),
        "prediction_not_modified": sha256_file(prediction_path)
        == prediction_seal["prediction"]["sha256"],
        "claim_scope_disclosed": bool(CLAIM_SCOPE),
    }
    failed = [key for key, value in evaluation_assertions.items() if not bool(value)]
    if failed:
        raise AssertionError(f"final evaluation hard gate failed: {failed}")

    evaluation_seal = {
        "status": "PRIVATE_EVALUATION_COMPLETE",
        "schema_version": "dcc-m3-v4-router-private-evaluation-seal-v2",
        "evaluation_id": EVALUATION_ID,
        "claim_scope": CLAIM_SCOPE,
        "test_use_policy": TEST_USE_POLICY,
        "prediction_run_id": prediction_seal["prediction_run_id"],
        "winner_model_id": prediction_seal["winner_model_id"],
        "winner_kind": prediction_seal["winner_kind"],
        "architecture_fingerprint": prediction_seal[
            "architecture_fingerprint"
        ],
        "routing_contract": prediction_seal["routing_contract"],
        "routing_sha256": prediction_seal["routing_sha256"],
        "checkpoint_bundle_sha256": prediction_seal[
            "checkpoint_bundle_sha256"
        ],
        "route_counts": prediction_seal["route_counts"],
        "evaluation_plan": plan_descriptor,
        "evaluation_plan_sha256": plan_descriptor["sha256"],
        "evaluation_spec_sha256": canonical_sha256(evaluation_spec),
        "prediction_seal": prediction_seal_descriptor,
        "prediction_seal_sha256": verification["prediction_seal_sha256"],
        "selection_seal_sha256": verification["selection_seal_sha256"],
        "inference_image_seal_sha256": verification[
            "inference_image_seal_sha256"
        ],
        "prediction_sha256": prediction_seal["prediction"]["sha256"],
        "test_eval_metadata_sha256": private_descriptor["sha256"],
        "private_eval_contract": private_contract_descriptor,
        "private_contract_sha256": private_contract_descriptor["sha256"],
        "private_contract_payload_sha256": canonical_sha256(private_contract),
        "private_contract_columns": private_contract["columns"],
        "dataset_seal_sha256": verification["dataset_seal_sha256"],
        "dataset_test_contract_sha256": verification[
            "dataset_test_contract_sha256"
        ],
        "test_sample_id_sha256": prediction_seal["test_sample_id_sha256"],
        "row_count": len(joined),
        "group_bootstrap_column": group_column,
        "metrics": metric_payload(metrics),
        "router_diagnostics": {
            "fallback_count": int(joined["fallback_used"].astype(bool).sum()),
            "fallback_rate_within_yolo_route": float(
                joined.loc[
                    joined["route_component"].eq("yolo_roi"),
                    "fallback_used",
                ].mean()
            ),
            "selected_error_case_count": int(len(error_selection)),
            "slice_count": int(len(performance_slices)),
        },
        "obstacle_auxiliary": {
            row["obstacle"]: {
                "n_known": int(row["n_known"]),
                "n_positive": int(row["n_positive"]),
                "threshold": json_safe_number(row.get("threshold")),
                "average_precision": json_safe_number(row.get("average_precision")),
                "f1": json_safe_number(row.get("f1")),
            }
            for row in aux_metrics.to_dict(orient="records")
        },
        "artifacts": artifact_descriptors,
        "assertions": evaluation_assertions,
    }
    evaluation_descriptor = upload_json_create_only(
        bucket,
        evaluation_seal,
        f"{EVALUATION_PREFIX}/evaluation_seal.json",
        "m3_v4_router_private_evaluation_seal",
    )
    print("PRIVATE EVALUATION COMPLETE")
    print("winner:", prediction_seal["winner_model_id"])
    print("overall RMSE (m):", overall_rmse)
    print("evaluation seal:", evaluation_descriptor["uri"])
    print("claim scope:", CLAIM_SCOPE)


## Entrypoint.  Run the two phases in separate Colab runtimes.


In [ ]:
if __name__ == "__main__":
    if RUN_PHASE == "INFERENCE":
        run_inference()
    else:
        run_evaluation()


EVALUATION PLAN FROZEN — private truth download is now allowed
plan seal: gs://YOUR_GCS_BUCKET/dcc/eval_private/mission3/v4/m3-v4-router-obstacle-eval-v4/evaluation_plan_seal.json


/tmp/ipykernel_1380/3292503775.py:367: UserWarning: Glyph 53441 (\N{HANGUL SYLLABLE TAG}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_1380/3292503775.py:367: UserWarning: Glyph 47448 (\N{HANGUL SYLLABLE RYU}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_1380/3292503775.py:367: UserWarning: Glyph 54364 (\N{HANGUL SYLLABLE PYO}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_1380/3292503775.py:367: UserWarning: Glyph 47732 (\N{HANGUL SYLLABLE MYEON}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_1380/3292503775.py:367: UserWarning: Glyph 49552 (\N{HANGUL SYLLABLE SON}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_1380/3292503775.py:367: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_1380/3292503775.py:367: UserWarning: Glyph 50752 (\N{HANGUL SYLLABLE WA}) missing from font(s) DejaVu Sans.
  fi

PRIVATE EVALUATION COMPLETE
winner: v4-fixed-router-multitask
overall RMSE (m): 0.14233391569554288
evaluation seal: gs://YOUR_GCS_BUCKET/dcc/eval_private/mission3/v4/m3-v4-router-obstacle-eval-v4/evaluation_seal.json
claim scope: internal_fixed_holdout_after_prior_truth_exposure
